In [1]:
from IPython.display import display, Markdown, HTML, clear_output, FileLink
from docx import Document
from docx.shared import Cm, Pt, Length, RGBColor
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT, WD_BREAK
from docx.enum.table import WD_ALIGN_VERTICAL, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement, parse_xml
from docx.oxml.ns import nsdecls
import docx.oxml.shared
import pandas as pd
import json
import re
from Bio import Entrez
import xmltodict
import urllib.request
from urllib.request import urlopen
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import os
import numpy as np
import openpyxl
import time
import itertools
import urllib.parse  
from decimal import Decimal
import warnings
warnings.filterwarnings('ignore')

#Other notebook involved
%run Redmine_Access.ipynb
%run MyCancerGenome_parsing_final.ipynb
%run ACMG.ipynb
%run "/home/ubuntu/jupyter_notebooks/Internal/Clinvar/Clinvar_annotation.ipynb"

#In house database File Paths
DB_PATH='In-House_Databases/'
NCBI_INFO_DB_FILE = '19K_Gene_GC_NCBI_Info.xlsx'
CKB_CORE_DB_FILE = 'Running Copy of Gene description_MyCancerGenome_CkBcore.xlsx'
PROGNOSIS_DB_FILE = 'V4 Running Copy of Information related to prognosis.xlsx'
DRUG_RESISTENCE_DB_FILE = 'V4 Running Copy of Drug resistance information.xlsx'
IMMUNOTHERAPIES_DB_FILE = 'V2 running FDA-approved cancer Immunotherapies Jan 2024.xlsx'


#INPUT FILE PATHS AND INPUT CUTOFF DATAS
ONCOMINE_REPORT='MANJU_DATTA_v1_CGP_Oncomine report.tsv'
FILTERED_IN='MANJU_DATTA_v1_CGP_c1871_2024-05-29-16-19-10-635-2024-05-31-06-22-15-058.tsv'
FILTERED_OUT='MANJU_DATTA_v1_CGP_c1871_2024-05-29-16-19-10-635-2024-05-31-06-22-05-779.tsv'
REDMINE_PATIENT_ID = '182'

GCHR='37'
TMB_CUTOFF = 10
MSI_CUTOFF = 25
SIFT_CUTOFF = 0.05
POLYPHEN_MIN = 0.15
POLYPHEN_MAX = 0.85

'''
ONCOMINE_REPORT='KIRAN_DONGAONKAR_RUN10_CGP_Oncomine report_Gastric cancer.tsv'
FILTERED_IN='KIRAN_DONGAONKAR_RUN10_CGP_c30596_2024-05-16-11-15-34-169-2024-05-17-12-31-55-105.tsv'
FILTERED_OUT='KIRAN_DONGAONKAR_RUN10_CGP_c30596_2024-05-16-11-15-34-169-2024-05-17-12-32-08-209.tsv'
REDMINE_PATIENT_ID = '177'
GCHR='37'

ONCOMINE_REPORT='SWARNALATA_PARIDA_RUN10_Oncomine report.tsv'
FILTERED_IN='SWARNALATA_PARIDA_RUN10_CGP_c35946_2024-05-16-11-17-31-750-2024-05-17-14-12-32-415.tsv'
FILTERED_OUT='SWARNALATA_PARIDA_RUN10_CGP_c35946_2024-05-16-11-17-31-750-2024-05-17-14-13-02-434.tsv'
REDMINE_PATIENT_ID = '179'
GCHR='37'

ONCOMINE_REPORT='AISHWARYA_BISWAS_Oncomine Report_Colorectal Cancer_txt.tsv'
FILTERED_IN='AISHWARYA_BISWAS_v1_CGP_c7320_2024-05-30-13-01-22-553-2024-05-30-12-23-11-628.tsv'
FILTERED_OUT='AISHWARYA_BISWAS_v1_CGP_c7320_2024-05-30-13-01-22-553-2024-05-30-12-23-29-241.tsv'
REDMINE_PATIENT_ID = '187'
GCHR='37'
'''

"\nONCOMINE_REPORT='KIRAN_DONGAONKAR_RUN10_CGP_Oncomine report_Gastric cancer.tsv'\nFILTERED_IN='KIRAN_DONGAONKAR_RUN10_CGP_c30596_2024-05-16-11-15-34-169-2024-05-17-12-31-55-105.tsv'\nFILTERED_OUT='KIRAN_DONGAONKAR_RUN10_CGP_c30596_2024-05-16-11-15-34-169-2024-05-17-12-32-08-209.tsv'\nREDMINE_PATIENT_ID = '177'\nGCHR='37'\n\nONCOMINE_REPORT='SWARNALATA_PARIDA_RUN10_Oncomine report.tsv'\nFILTERED_IN='SWARNALATA_PARIDA_RUN10_CGP_c35946_2024-05-16-11-17-31-750-2024-05-17-14-12-32-415.tsv'\nFILTERED_OUT='SWARNALATA_PARIDA_RUN10_CGP_c35946_2024-05-16-11-17-31-750-2024-05-17-14-13-02-434.tsv'\nREDMINE_PATIENT_ID = '179'\nGCHR='37'\n\nONCOMINE_REPORT='AISHWARYA_BISWAS_Oncomine Report_Colorectal Cancer_txt.tsv'\nFILTERED_IN='AISHWARYA_BISWAS_v1_CGP_c7320_2024-05-30-13-01-22-553-2024-05-30-12-23-11-628.tsv'\nFILTERED_OUT='AISHWARYA_BISWAS_v1_CGP_c7320_2024-05-30-13-01-22-553-2024-05-30-12-23-29-241.tsv'\nREDMINE_PATIENT_ID = '187'\nGCHR='37'\n"

In [2]:
biomarkers_df = pd.DataFrame() # comes from readOncomineReport()
hrr_df= pd.DataFrame() # comes from readOncomineReport()
dna_sequence_variants_df = pd.DataFrame() # comes from readOncomineReport()
fusion_variants_df = pd.DataFrame() # comes from readOncomineReport()
copy_number_variations_df = pd.DataFrame() # comes from readOncomineReport()
dna_sequence_variants_merged_with_filter_df = pd.DataFrame() # comes from merge_filtered_variant_data
fusion_merged_with_filter_df = pd.DataFrame() # comes from merge_filtered_variant_data
copy_number_variations_merged_with_filter_df = pd.DataFrame() # comes from merge_filtered_variant_data
snv_df = pd.DataFrame() # comes from process_dna_sequence_variants()
cnv_df = pd.DataFrame() # comes from process_copy_number_variants()
fusions_df = pd.DataFrame() # comes from process_fusion_variants()
cancer_type = '' # comes from readOncomineReport()
sample_id = '' # comes from readOncomineReport()
mdc = '' # comes from readOncomineReport()
tbc = '' # comes from readOncomineReport()
msi_score = '' # comes from readOncomineReport()
mutational_burden = '' # comes from readOncomineReport()
tmb_status_score = '' # comes from report_summary(document)
msi_value = '' # comes from report_summary(document)
loh_score='' # comes from report_summary(document)
mmr_status = '' # comes from immunotherapy_markers(document)

In [3]:
# Reads the input file line wise and extracts data 
def readOncomineReport():
    #Global variables that will be changed in this method
    global biomarkers_df,hrr_df,dna_sequence_variants_df,fusion_variants_df,copy_number_variations_df,cancer_type,sample_id,mdc,tbc,msi_score,mutational_burden
    with open(ONCOMINE_REPORT, 'r', encoding='latin1') as file:
        lines = file.readlines()
        for i in range(len(lines)):
            line = lines[i].strip()
            if line.startswith("Sample Cancer Type"):
                cancer_type = line.split('\t')[1]
            elif line.lower().startswith("**sample id:**|"):
                sample_id = line.split('|')[1].strip()
            elif line.lower().startswith('mean depth coverage'):
                mdc = line.split(":")[1].strip()
            elif line.lower().startswith('target base coverage'):
                tbc = line.split(":")[1].strip()
            elif line.startswith("MSI Score"):
                msi_score = line.split(":")[1].strip()
            elif line.startswith("TMB Score"):
                mutational_burden = line.split(":")[1].strip()
            elif line == "Relevant Biomarkers":
                #Ignore next blank line
                i=i+1
                data=[]
                while True:
                    i=i+1
                    line=lines[i].strip()
                    if line:
                        data.append(line.split('\t'))
                    else:
                        break
                #print(data)
                biomarkers_df = pd.DataFrame(data)
                headers = biomarkers_df.iloc[0].values
                biomarkers_df.columns = headers
                biomarkers_df.drop(index=0, axis=0, inplace=True)
                #print('-----------biomarkers_df------------')
                #display(biomarkers_df)
            elif line == "HRR Details":
                #Ignore next blank line
                i=i+1
                data=[]
                while True:
                    i=i+1
                    line=lines[i].strip()
                    if line:
                        data.append(line.split('\t'))
                    else:
                        break
                #print(data)
                hrr_df = pd.DataFrame(data)
                headers = hrr_df.iloc[0].values
                hrr_df.columns = headers
                hrr_df.drop(index=0, axis=0, inplace=True)
                #print('-----------hrr_df------------')
                #display(hrr_df)
            elif line == "DNA Sequence Variants":
                #Ignore next blank line
                i=i+1
                data=[]
                while True:
                    i=i+1
                    line=lines[i].strip()
                    if line:
                        data.append(line.split('\t'))
                    else:
                        break
                #print(data)
                dna_sequence_variants_df = pd.DataFrame(data)
                headers = dna_sequence_variants_df.iloc[0].values
                dna_sequence_variants_df.columns = headers
                dna_sequence_variants_df.drop(index=0, axis=0, inplace=True)
                #print('-----------dna_sequence_variants_df------------')
                #display(dna_sequence_variants_df)
            elif line == "Gene Fusions":
                #Ignore next blank line
                i=i+1
                data=[]
                while True:
                    i=i+1
                    line=lines[i].strip()
                    #print(line)
                    #print(line.split('\t'))
                    if line:
                        data.append(line.split('\t'))
                    else:
                        break
                #print(data)
                fusion_variants_df = pd.DataFrame(data)
                headers = fusion_variants_df.iloc[0].values
                fusion_variants_df.columns = headers
                fusion_variants_df.drop(index=0, axis=0, inplace=True)
                #print('-----------fusion_variants_df------------')
                #display(fusion_variants_df)
            elif line == "Copy Number Variations":
                #Ignore next blank line
                i=i+1
                data=[]
                while True:
                    i=i+1
                    line=lines[i].strip()
                    if line:
                        data.append(line.split('\t'))
                    else:
                        break
                #print(data)
                copy_number_variations_df = pd.DataFrame(data)
                headers = copy_number_variations_df.iloc[0].values
                copy_number_variations_df.columns = headers
                copy_number_variations_df.drop(index=0, axis=0, inplace=True)
                copy_number_variations_df = copy_number_variations_df.drop(columns=['Tier', 'Filter', 'Oncomine Gene Class'])
    
                #print('-----------copy_number_variations_df------------')
                #display(copy_number_variations_df)
            elif line == "Biomarker Descriptions":
                break  

In [4]:
#Read Filtered-in and Filtered-out files
def read_and_preprocess_data(file_path):
    # Open the file at the given file_path for reading
    with open(file_path, 'r') as file:
        variant_group = None  # Initialize the variant_group variable
        lines = []  
        # Iterate through each line in the file
        for line in file:
            # Check if the line starts with '##variantGroup='
            if line.startswith('##variantGroup='):
                # If yes, extract the variant group information and assign it to the variant_group variable
                variant_group = line.split('=')[1].strip()
            # Check if the line does not start with '##'
            elif not line.startswith('##'):
                # If yes, append the line to the lines list (excluding lines starting with '##')
                lines.append(line.strip())
    
    # Convert the list of lines into a DataFrame, splitting each line by the tab ('\t') character
    df = pd.DataFrame([line.split('\t') for line in lines])
    # Set the column names of the DataFrame to the values in the first row
    df.columns = df.iloc[0]
    df = df.rename(columns={'Observed Allele': 'Alt'})
    # Exclude the first row (header row) from the DataFrame
    df = df[1:]
    return df, variant_group  

# Read and preprocess the data, and get variant group
def process_filtered_data():
    filtered_in_df, variant_group_in = read_and_preprocess_data(FILTERED_IN)
    filtered_out_df, variant_group_out = read_and_preprocess_data(FILTERED_OUT)
    
    #Columns extracted from filtered_in and filtered_out file                                                        
    selected_columns = ['Locus', 'Filter', 'Genes (Exons)', 'Exon', 'Length', 'Oncomine Gene Class', 'Variant ID', 'CytoBand', 'ExAC GAF', 'Allele Coverage', 'Ref', 'Alt', 'dbSNP', 'SIFT', 'PolyPhen']
    
    # Assign to appropriate dataframe based on the variant group.
    #If the file in the line ##variantGroup= has DEFAULT it is a filtered in file. If the file has FILTERED_OUT it is a filtered out file
    if variant_group_in == 'DEFAULT':
        filtered_in_df = filtered_in_df[selected_columns]
    elif variant_group_in == 'FILTERED_OUT':
        filtered_out_df = filtered_out_df[selected_columns]
    if variant_group_out == 'DEFAULT':
        filtered_in_df = filtered_in_df[selected_columns]
    elif variant_group_out == 'FILTERED_OUT':
        filtered_out_df = filtered_out_df[selected_columns]
    return filtered_in_df, filtered_out_df

#Merge dna_sequence_variants_df with filtered_in and filtered_out files
def merge_dna_sequence_variants(filtered_in_df, filtered_out_df):
    global dna_sequence_variants_df
    
    # Check if the global dataframe is empty, and if so, return an empty dataframe
    if dna_sequence_variants_df.empty:
        return pd.DataFrame()

    # Merge the global dataframe with the filtered_in_df on the 'Locus' column, keeping only the matches (inner join)
    merged_with_filter_in_df = dna_sequence_variants_df.merge(filtered_in_df, on='Locus', how='inner', suffixes=('', '_y'))
    # Drop the duplicate columns created by the merge (columns ending with '_y')
    merged_with_filter_in_df.drop(merged_with_filter_in_df.filter(regex='_y$').columns, axis=1, inplace=True)
    
    # Merge the global dataframe with the filtered_out_df on the 'Locus' column, keeping only the matches (inner join)
    merged_with_filter_out_df = dna_sequence_variants_df.merge(filtered_out_df, on='Locus', how='inner', suffixes=('', '_y'))
    # Drop the duplicate columns created by the merge (columns ending with '_y')
    merged_with_filter_out_df.drop(merged_with_filter_out_df.filter(regex='_y$').columns, axis=1, inplace=True)
    
    # Concatenate the two merged dataframes (filtered_in and filtered_out merges)
    merged_with_filter_df = pd.concat([merged_with_filter_in_df, merged_with_filter_out_df])
    merged_with_filter_df = merged_with_filter_df.drop_duplicates(subset=['Locus'])
    # Filter out rows where the 'Filter' column has values 'NOCALL', 'FAIL', or 'NO CALL'
    merged_with_filter_df = merged_with_filter_df[~(merged_with_filter_df['Filter'].isin(['NOCALL', 'FAIL', 'NO CALL']))]
    merged_with_filter_df.reset_index(drop=True, inplace=True)

    # Add a new column 'Allele Coverage Alt' initialized with missing values
    merged_with_filter_df['Allele Coverage Alt'] = pd.NA
    for index, row in merged_with_filter_df.iterrows():
        observed_allele = row['Alt']  # Get the observed allele
        coverage_data = str(row['Allele Coverage'])  # Convert the 'Allele Coverage' to string
        observed_coverage = None  # Initialize observed coverage as None
        # If the coverage data contains multiple values, split and process them
        if ', ' in coverage_data:
            for part in coverage_data.split(','):
                allele, coverage = part.split('=')  # Split the part into allele and coverage
                if allele.strip() == observed_allele:  # If the allele matches the observed allele
                    observed_coverage = int(coverage)  # Convert the coverage to an integer
                    break  # Stop once the matching coverage is found
        # If no observed coverage found, set it as missing
        if observed_coverage is None:
            observed_coverage = pd.NA
        # Update the 'Allele Coverage Alt' column for the current row
        merged_with_filter_df.at[index, 'Allele Coverage Alt'] = observed_coverage
    
    # Split the 'Locus' column to extract chromosome number and position
    split_data = merged_with_filter_df["Locus"].str[3:].str.split(":", expand=True)
    # Add 'Chr_no' column with the first part of the split data or 'NA' if not available
    merged_with_filter_df["Chr_no"] = split_data[0] if len(split_data.columns) >= 1 else 'NA'
    # Add 'Chr_pos' column with the second part of the split data or 'NA' if not available
    merged_with_filter_df["Chr_pos"] = split_data[1] if len(split_data.columns) >= 2 else 'NA'
    # Creating new columns for the table construction in the report 
    merged_with_filter_df['Genes_Locus']= merged_with_filter_df['Gene']+'\n'+merged_with_filter_df['Locus']
    merged_with_filter_df["Mutation_taster_query"] = "chromosome="+merged_with_filter_df["Chr_no"]+"&position="+merged_with_filter_df["Chr_pos"]+"&ref="+merged_with_filter_df["Ref"]+"&alt="+merged_with_filter_df["Alt"]
    merged_with_filter_df['Amino Acid Change'] = merged_with_filter_df['Amino Acid Change'].str.extract(r'p\.\(([A-Z0-9*fs]*)\)')
    merged_with_filter_df['Amino Acid Change'] = merged_with_filter_df['Amino Acid Change'].replace('?', np.nan).fillna('')
    merged_with_filter_df['Genomic alterations'] = merged_with_filter_df['Coding'] + '\n' + merged_with_filter_df['Amino Acid Change']
    merged_with_filter_df['Genes_codings'] = merged_with_filter_df['Gene'] + '\n' + merged_with_filter_df['Coding']
    
    return merged_with_filter_df  


#Merge fusion_variants_df with filtered_in and filtered_out files
def merge_fusion_variants(filtered_in_df, filtered_out_df):
    global fusion_variants_df
    
    # Check if the global dataframe is empty, if so, return an empty dataframe
    if fusion_variants_df.empty:
        return pd.DataFrame()

    # Merge the global dataframe with the filtered_in_df on the 'Locus' column, keeping only the matches (inner join)
    merged_with_filter_in_df = fusion_variants_df.merge(filtered_in_df, on='Locus', how='inner', suffixes=('', '_y'))
    # Drop the duplicate columns created by the merge (columns ending with '_y')
    merged_with_filter_in_df.drop(merged_with_filter_in_df.filter(regex='_y$').columns, axis=1, inplace=True)
    
    # Merge the global dataframe with the filtered_out_df on the 'Locus' column, keeping only the matches (inner join)
    merged_with_filter_out_df = fusion_variants_df.merge(filtered_out_df, on='Locus', how='inner', suffixes=('', '_y'))
    # Drop the duplicate columns created by the merge (columns ending with '_y')
    merged_with_filter_out_df.drop(merged_with_filter_out_df.filter(regex='_y$').columns, axis=1, inplace=True)
    
    # Concatenate the two merged dataframes (filtered_in and filtered_out merges)
    merged_with_filter_df = pd.concat([merged_with_filter_in_df, merged_with_filter_out_df])
    merged_with_filter_df = merged_with_filter_df.drop_duplicates(subset=['Locus'])
    # Filter out rows where the 'Filter' column has values 'NOCALL', 'FAIL', or 'NO CALL'
    merged_with_filter_df = merged_with_filter_df[~(merged_with_filter_df['Filter'].isin(['NOCALL', 'FAIL', 'NO CALL']))]
    return merged_with_filter_df  

    
#Merge copy_number_variations_df with filtered_in and filtered_out files
def merge_copy_number_variations(filtered_in_df, filtered_out_df):
    global copy_number_variations_df
    
    # Check if the global dataframe is empty, if so, return an empty dataframe
    if copy_number_variations_df.empty:
        return pd.DataFrame()

    # Merge the global dataframe with the filtered_in_df on the 'Locus' column, keeping only the matches (inner join)
    merged_with_filter_in_df = copy_number_variations_df.merge(filtered_in_df, on='Locus', how='inner', suffixes=('', '_y'))
    # Drop the duplicate columns created by the merge (columns ending with '_y')
    merged_with_filter_in_df.drop(merged_with_filter_in_df.filter(regex='_y$').columns, axis=1, inplace=True)
    
    # Merge the global dataframe with the filtered_out_df on the 'Locus' column, keeping only the matches (inner join)
    merged_with_filter_out_df = copy_number_variations_df.merge(filtered_out_df, on='Locus', how='inner', suffixes=('', '_y'))
    # Drop the duplicate columns created by the merge (columns ending with '_y')
    merged_with_filter_out_df.drop(merged_with_filter_out_df.filter(regex='_y$').columns, axis=1, inplace=True)
    
    # Concatenate the two merged dataframes (filtered_in and filtered_out merges)
    merged_with_filter_df = pd.concat([merged_with_filter_in_df, merged_with_filter_out_df])
    merged_with_filter_df = merged_with_filter_df.drop_duplicates(subset=['Locus'])
    # Filter out rows where the 'Filter' column has values 'NOCALL', 'FAIL', or 'NO CALL'
    merged_with_filter_df = merged_with_filter_df[~(merged_with_filter_df['Filter'].isin(['NOCALL', 'FAIL', 'NO CALL']))]

    return merged_with_filter_df  


# Merge variant data with filtered_in and filtered_out files for DNA sequence variants, fusion variants, and copy number variations.
def merge_filtered_variant_data():
    global dna_sequence_variants_merged_with_filter_df, fusion_merged_with_filter_df, copy_number_variations_merged_with_filter_df
    filtered_in_df, filtered_out_df = process_filtered_data()
    
    # DNA sequence variants dataframe 
    dna_sequence_variants_merged_with_filter_df = merge_dna_sequence_variants(filtered_in_df, filtered_out_df)
    
    # Fusion variants dataframe 
    fusion_merged_with_filter_df = merge_fusion_variants(filtered_in_df, filtered_out_df)
    
    # Copy number variations dataframe
    copy_number_variations_merged_with_filter_df = merge_copy_number_variations(filtered_in_df, filtered_out_df)
    return dna_sequence_variants_merged_with_filter_df, fusion_merged_with_filter_df, copy_number_variations_merged_with_filter_df

In [5]:
""" calls mutation taster api
args: the dataframe with the Mutation_taster-query
returns: same dataframe with an added column with the mutation taster results


Mutation taster does not work for indels with the .tsv ref and alt data
this function leaves the results row empty in the case of an indel"""

def mutation_taster(df):
  df['Mutation_taster_results'] = pd.Series(dtype=str)
  base_url = "https://www.genecascade.org/MTc2021/ChrPos102.cgi"
  for index,row in df.iterrows():
    query =  row["Mutation_taster_query"]
    url = f"{base_url}?{query}"  # Use URL parameters instead of query string concatenation

    response = requests.get(url)
    prediction_text_list=[]

    # Check for successful response
    if response.status_code == 200:
        # Parse HTML content with BeautifulSoup
        soup = bs(response.text, 'html.parser')

        # Find the summary table
        summary_table = soup.find('table', class_='resultlinks')  # Assuming specific class for the table

        if summary_table is not None:
            # Initialize empty dictionary to store transcript-prediction pairs
            all_predictions = {}

            # Skip the header row (index 0)
            for row in summary_table.find_all('tr')[1:]:
                # Extract transcript ID from the first cell (assuming anchor tag)
                transcript_id_element = row.find('td')
                if transcript_id_element is not None:
                    transcript_id = transcript_id_element.text.strip()

                # Extract prediction from the third cell (assuming span tag)
                prediction_element = row.find_all('td')[2]
                if prediction_element is not None:
                    # Extract text from the prediction element's child (assuming it's a span)
                    prediction_text = prediction_element.find('span').text.strip()

                    # Add transcript-prediction pair to the dictionary
                    all_predictions[transcript_id] = prediction_text
                    #print(prediction_text)

            # Print extracted predictions
            if all_predictions:
                #print("Transcript ID - Prediction:")
                for transcript_id, prediction in all_predictions.items():
                    #print(f"{transcript_id}: {prediction}")
                    prediction_text_list.append(prediction)
                df.at[index, "Mutation_taster_results"] =  prediction_text_list
                

            else:
                print("No predictions found in the table.")
                df.at[index, "Mutation_taster_results"] =  []
        else:
            print("Summary table not found in the response.")
            df.at[index, "Mutation_taster_results"] =  []

        #print(prediction_text_list)
    else:
        print(f"Error: Failed to fetch URL: {url} (Status code: {response.status_code})")
        df.at[index, "Mutation_taster_results"] =  []
  return df

#dna_sequence_variants_df["Mutation_taster_results"]= all_predictions_list

In [6]:
def clinvar_extraction(input_df,gref):
    """
    Extracts ClinVar annotations for variants in the input DataFrame.

    Returns:
    - DataFrame: Input DataFrame with added ClinVar annotations.

    This function iterates over each variant in the input DataFrame, checks if ClinVar annotations
    are already cached, and retrieves them if available. If not cached, it calls appropriate functions
    based on the length of reference and alternate alleles to obtain ClinVar annotations. The results
    are then added to the input DataFrame. For indels, ClinVar annotations cannot be obtained, and
    thus the respective rows remain empty in the output DataFrame.
    """
    global dna_sequence_variants_merged_with_filter_df
    #making a copy of the input file dataframe to add the intermediate processes columns
    input_df_copy=input_df.copy()
    dictionary_df=pd.DataFrame(columns=["Locus:ref:alt"])

    for index, row in input_df_copy.iterrows():
        output_df=pd.DataFrame()
        chr_no = row['Chr_no']
        chr_pos = row['Chr_pos']
      #coding = row["coding"]
        ref = row['Ref']
        alt = row['Alt']
        check=f"{chr_no}:{chr_pos}:{ref}:{alt}"
        #print (check)
        #print(list(dictionary_df["Locus:ref:alt"]))
        if check in list(dictionary_df["Locus:ref:alt"]):
            #print("using cached data")
            cached_row = dictionary_df[dictionary_df["Locus:ref:alt"] == check].iloc[0]
            # Update input_df with cached data
            input_df.loc[index, 'Clinvar_accession_no'] = cached_row['Clinvar_accession_no']
            input_df.loc[index, 'Clinvar_Protein_change'] = cached_row['ClinVar_Protein_change']
            input_df.loc[index, 'gnomAD_value'] = cached_row['gnomAD_value']
            input_df.loc[index, '1000_Genomes_value'] = cached_row['1000_Genomes_value']
            input_df.loc[index, 'ExAC_value'] = cached_row['ExAC_value']
            input_df.loc[index, 'dbSNP_id'] = cached_row['dbSNP_id']
            input_df.loc[index, 'OMIM_id'] = cached_row['OMIM_id']
            input_df.loc[index, 'Germline_classification'] = cached_row['Germline_classification']
            input_df.loc[index, 'Somatic_clinical_impact'] = cached_row['Somatic_clinical_impact']
            input_df.loc[index, 'Clinvar_link'] = cached_row['Clinvar_link']
        else:
            if len(ref)==len(alt)==1:
                output_df=clinvar_ref_alt_input(chr_no,chr_pos,ref,alt,gref)
                output_df.loc[0,"Locus:ref:alt"]= f"{chr_no}:{chr_pos}:{ref}:{alt}"
                dictionary_df=pd.concat([dictionary_df, output_df], axis=0)
                #display(dictionary_df)
            else:
                ref=ref.strip()
                alt=alt.strip()
                output_df= clinvar_ref_alt_input_indel(chr_no,chr_pos,ref,alt,gref)
                output_df.loc[0,"Locus:ref:alt"]= f"{chr_no}:{chr_pos}:{ref}:{alt}"
                dictionary_df=pd.concat([dictionary_df, output_df], axis=0)
            if not output_df.empty:
                #Adding the clinvar annotation columns to the original input file dataframe
                input_df.loc[index, 'Clinvar_accession_no'] = output_df.loc[0,'Clinvar_accession_no']
                input_df.loc[index, 'Clinvar_Protein_change'] = output_df.loc[0,'ClinVar_Protein_change']
                input_df.loc[index, 'gnomAD_value'] = output_df.loc[0,'gnomAD_value']
                input_df.loc[index, '1000_Genomes_value'] = output_df.loc[0,'1000_Genomes_value']
                input_df.loc[index, 'ExAC_value'] = output_df.loc[0,'ExAC_value']
                input_df.loc[index, 'dbSNP_id'] = output_df.loc[0,'dbSNP_id']
                input_df.loc[index, 'OMIM_id'] = output_df.loc[0,'OMIM_id']
                input_df.loc[index, 'Germline_classification'] = output_df.loc[0,'Germline_classification']
                input_df.loc[index, 'Somatic_clinical_impact'] = output_df.loc[0,'Somatic_clinical_impact']
                input_df.loc[index, 'Clinvar_link'] = output_df.loc[0,'Clinvar_link']
    input_df['Clinvar_Protein_change']= input_df['Clinvar_Protein_change'].fillna("")
#display(dna_sequence_variants_merged_with_filter_df)

In [7]:
#Writing of the document
#Adds a custom header and footer to each section of a document.
def add_header_footer(document):
    print('Adding Header and Footer')
    for section in document.sections:
        # Set header
        header = section.header
        header_paragraph = header.paragraphs[0] if header.paragraphs else header.add_paragraph()
        header_paragraph.paragraph_format.left_indent = Cm(2.7) 
        image_path = os.path.join("Heading_Images", "G-KnowMe_Logo.png")
        header_run = header_paragraph.add_run()
        header_run.add_picture(image_path, width=Cm(5.4), height=Cm(2.71))
        
        # Set footer
        footer = section.footer
        footer_paragraph = footer.paragraphs[0] if footer.paragraphs else footer.add_paragraph()
        footer_paragraph.paragraph_format.left_indent = Cm(2.7)  
        footer_run = footer_paragraph.add_run("Private and confidential")
        footer_run.font.size = Pt(12)
        footer_run.font.name = 'Times New Roman'
        footer_run.italic = True

In [8]:
# Adds an internal reference table to the document.
def add_internal_reference_table(document):
    global biomarkers_df  
    print('Adding Internal refernce tables')
    # Adjust margins for all sections
    sections = document.sections
    for section in sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)

    # Add centered heading
    centered_heading = document.add_paragraph("Table only for internal reference")
    centered_heading.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    centered_heading.runs[0].font.size = Pt(12)
    centered_heading.runs[0].font.name = 'Times New Roman'
    
    # Define table headings
    headings = ["Patient name", "Age", "Gender", "Diagnosis", "Main tumor type", "Tumor subtype", "Comorbidities", "Specimen",
                "Histopathology in case of breast cancer/ovarian cancer etc.", "Metastatic status",
                "Known Infectious causes (for example HPV)?", "Treatment and progression/resistance history",
                "TMB status", "MSI status", "PD-L1 status", "HRR status", "MMR status",
                "Genes for clinical interpretation", "Additional comment/Family history etc."]
    
    # Add table with predefined headings
    table = document.add_table(rows=0, cols=2)
    table.style = 'Table Grid'
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(5.26)
    table.columns[1].width = Cm(12.7)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    
    # Add headings to the table
    for heading in headings:
        row = table.add_row().cells
        row[0].text = heading
        row[1].text = ""
        for cell in row:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.name = 'Times New Roman'
                    run.font.size = Pt(12)
    
    # Add one line spacing after the table
    document.add_paragraph('').add_run().add_break(WD_BREAK.LINE)
    
    if biomarkers_df is not None:  
        # Add biomarkers_df table
        column_widths = [28 / biomarkers_df.shape[1]] * biomarkers_df.shape[1]
        biomarkers_table = document.add_table(biomarkers_df.shape[0] + 1, biomarkers_df.shape[1])
        biomarkers_table.style = 'Table Grid'
        biomarkers_table.autofit = False
        biomarkers_table.allow_autofit = False
        biomarkers_table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        
        # Set the width for each column
        for j, width in enumerate(column_widths):
            biomarkers_table.columns[j].width = Cm(width)
        
        # Add column headings
        for j, col in enumerate(biomarkers_df.columns):
            cell = biomarkers_table.cell(0, j)
            cell.text = col
            cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
            cell.paragraphs[0].runs[0].font.size = Pt(10)
            cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        
        # Add data to the table
        for i in range(biomarkers_df.shape[0]):
            for j in range(biomarkers_df.shape[1]):
                cell = biomarkers_table.cell(i + 1, j)
                cell.text = str(biomarkers_df.iloc[i, j])
                cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
                cell.paragraphs[0].runs[0].font.size = Pt(9)
    else:
        # If biomarkers_df is None, add a message indicating no relevant data
        document.add_paragraph('NO RELEVANT BIOMARKERS TABLE GIVEN')

    # Add page break after the table
    document.add_page_break()

In [9]:
#Creates a patient information table and gets added when patient_information(document) is called. This information is taken from Redmine 
def patient_information_table():
    global patient_info
    print('Patient Information table added')
    sections = doc.sections
    for section in sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
    FFPE_block_number = patient_info.get('FFPE_block_number', '') 
    FFPE = 'FFPE Block ' + '( ' + FFPE_block_number + ')' if FFPE_block_number else ''
    table = doc.add_table(rows=1, cols=2)
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(5.1)
    table.columns[1].width = Cm(11.89)
    table.style = 'Table Grid'
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.cell(0, 0).text = 'Name'
    table.cell(0, 1).text = patient_info.get('patient_name', '')or '' 
    table.add_row().cells[0].text = 'Age/Gender'
    table.cell(1, 1).text = f"{patient_info.get('patient_age', '')} years /{patient_info.get('patient_gender', '')}"or '' 
    table.add_row().cells[0].text = 'Referring Physician:'
    table.cell(2, 1).text = patient_info.get('referring_physician', '')or '' 
    table.add_row().cells[0].text = 'Referring Centre'
    table.cell(3, 1).text = patient_info.get('referring_centre', '')or '' 
    table.add_row().cells[0].text = 'Specimen Type'
    table.cell(4, 1).text = FFPE
    table.add_row().cells[0].text = 'Sample ID'
    table.cell(5, 1).text = patient_info.get('sample_id', '')or '' 
    table.add_row().cells[0].text = 'Patient ID'
    table.cell(6, 1).text = patient_info.get('patient_id', '')or '' 
    table.add_row().cells[0].text = 'Date received'
    table.cell(7, 1).text = patient_info.get('date_received', '')or '' 
    table.add_row().cells[0].text = 'Report Date'
    table.cell(8, 1).text = patient_info.get('report_date', '')or ''
    
    font_style_column1 = 'Times New Roman'
    font_size_column1 = Pt(12)
    bold_column1 = True
    font_style_column2 = 'Times New Roman'
    font_size_column2 = Pt(12)
    bold_column2 = False
    for row in table.rows:
        row.cells[0].paragraphs[0].runs[0].font.name = font_style_column1
        row.cells[0].paragraphs[0].runs[0].font.size = font_size_column1
        row.cells[0].paragraphs[0].runs[0].bold = bold_column1
    for row in table.rows:
        row.cells[1].paragraphs[0].runs[0].font.name = font_style_column2
        row.cells[1].paragraphs[0].runs[0].font.size = font_size_column2
        row.cells[1].paragraphs[0].runs[0].bold = bold_column2

In [10]:
def patient_information(document):
    """
    Adds patient information section to the document.

    This function adds patient information section including an image and a table.

    """
    print('In Patient Information Section')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(7.69)
    new_height = Cm(0.8)
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", "Patient_information.png")
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    doc = Document()
    p = doc.add_paragraph()
    p.paragraph_format.line_spacing = 1
    p.paragraph_format.space_after = 12
    patient_information_table()

In [11]:
def oncoprecise_image(document):
    p = document.add_paragraph()
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    new_width = Cm(14.31)
    new_height = Cm(1.27)
    image_path = os.path.join("Heading_Images", 'Oncoprecise.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)

In [12]:
def clinical_history(document):
    """
    Adds clinical history section to the document.

    This function includes an image and textual information about the patient's clinical history.

    """
    global patient_info  
    print('In Clinical History section')

    # Set margins for all sections
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    
    # Define dimensions for the image
    new_width = Cm(6.73)
    new_height = Cm(0.63)
    
    # Add image to the document
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Clinical_history.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    
    # Add textual information about clinical history
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    #This information is taken from Redmine
    main_tumor_types = patient_info.get('Main Tumor Type', '')
    p.add_run(main_tumor_types)
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'

In [13]:
def report_summary_table(document, table_data):
    print('Adding Report summary Table')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
    table = document.add_table(rows=1, cols=2)
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(7.51)
    table.columns[1].width = Cm(9.97)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style = 'Table Grid'
    first_row = table.rows[0].cells
    first_row[0].text = "Alteration Description"
    first_row[1].text = "Findings"
    for cell in first_row:
        cell.paragraphs[0].runs[0].bold = True
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        cell.paragraphs[0].runs[0].font.color.rgb = RGBColor(0, 0, 128) 
    row_height_cm = 1.04
    first_row = table.rows[0]
    first_row.height = Pt(row_height_cm * 28.3465)
    for key, value in table_data.items():
        row_cells = table.add_row().cells
        cell = row_cells[0]
        cell.text = key
        cell.paragraphs[0].runs[0].bold = True
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        if value is not None:  # Check if value is not None before assigning
            row_cells[1].text = value
            cell1 = row_cells[1]
            cell1.paragraphs[0].runs[0].font.name = 'Times New Roman'
            cell1.paragraphs[0].runs[0].font.size = Pt(12)
        else:
            row_cells[1].text = "N/A"  # If value is None, assign "N/A" to the cell
    return table

In [14]:
def report_summary(document):
    """
    Adds a report summary section to the document.

    This function includes an image, textual information, and a summary table about the patient's report.

    """
    print('In Report summary section')
    global patient_info, dna_sequence_variants_df, dna_sequence_variants_merged_with_filter_df, fusion_variants_df, fusion_merged_with_filter_df, copy_number_variations_df, copy_number_variations_merged_with_filter_df, mutational_burden, msi_score, hrr_df, tmb_status_score, msi_value, loh_score
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(5.82)
    new_height = Cm(0.66)
    paragraph = document.add_paragraph()
    image_path = os.path.join("Heading_Images", 'Report_summary.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2 * 0.42)
    
    #This information comes from Redmine
    FFPE=patient_info.get('FFPE_block_number', '')or '___' 
    tumor_cell=patient_info.get('tumor_cells', '')or '___'
    p.add_run("The test was performed on Block ")
    p.add_run(FFPE).bold = True
    p.add_run(" which showed presence of ")
    p.add_run(tumor_cell).bold = True
    p.add_run("% tumor cells.")    
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'

    # Determine TMB status score 
    if mutational_burden is not None:  # Check if mutational_burden exists
        # Extract numeric part from mutational_burden
        numeric_part = ''.join(c for c in mutational_burden if c.isdigit() or c == '.')
        try:
            if numeric_part: 
                mutational_burden_value = float(numeric_part)
                # If it's less than 10, the status is set to 'Low', otherwise, it's set to 'High'.
                mutational_burden_status = 'Low' if mutational_burden_value < TMB_CUTOFF else 'High'
                mutational_burden_status = 'TMB-' + mutational_burden_status + ' (' + mutational_burden + 'mut/Mb)'
            else:
                # If numeric_part is empty, set mutational burden status to Unknown
                mutational_burden_status = 'Unknown'
                mutational_burden_status = 'TMB-' + mutational_burden_status + ' (' + mutational_burden + 'mut/Mb)'
        except ValueError:
            mutational_burden_status = 'CONVERSION TO FLOAT FAILED - ERROR IN CODE'
    else:
        # If mutational_burden is None
        mutational_burden_status = 'NOT GIVEN'
    tmb_status_score = mutational_burden_status

    # Determine MSI score 
    if msi_score is not None:
        # Extract numeric part from MSI score
        numeric_part = ''.join(c for c in msi_score if c.isdigit() or c == '.') 
        try:
            if numeric_part:
                msi_score_value = float(numeric_part)
                # If it's less than 25, the status is set to 'Microsatellite stable (MSS)', otherwise, it's set to 'Microsatellite instability (MSI)'.
                msi_status = 'Microsatellite stable (MSS)' if msi_score_value < MSI_CUTOFF else 'Microsatellite instability (MSI)'
                msi_status = msi_status + ' (Score: ' + msi_score + ')'
            else:
                # If no numeric part found, set status to 'Unknown'
                msi_status = 'Unknown'
        except ValueError:
            msi_status = 'CONVERSION TO FLOAT FAILED - ERROR IN CODE'
    else:
        # If MSI score is None
        msi_status = 'NOT GIVEN'
    msi_value = msi_status
    
    # Determine LOH score 
    if hrr_df.empty:
        loh_score = 'NOT IDENTIFIED'
    else:
        # Extract LOH score from DataFrame if available
        loh_score = hrr_df.loc[hrr_df['Gene/Genomic Alteration'] == 'LOH percentage', 'Finding'].str.extract(r'(\d+\.\d+)').iloc[0, 0] if not hrr_df.loc[hrr_df['Gene/Genomic Alteration'] == 'LOH percentage', 'Finding'].empty else '0'
        # Append '%' to the LOH score if it's not None
        if loh_score is not None:
            loh_score = str(loh_score) + '%'
            loh_score_format = 'LOH (' + loh_score+ ')'
    
    # Process DNA sequence variants DataFrame
    if not dna_sequence_variants_df.empty:        
        dna_sequence_variants_df = dna_sequence_variants_merged_with_filter_df.copy()
        dna_sequence_variants_df['Gene_Coding'] = dna_sequence_variants_merged_with_filter_df['Gene'] + ' (' + dna_sequence_variants_merged_with_filter_df['Coding'] + ')'
        # Concatenate gene mutations into a string separated by newline characters
        gene_mutations = '\n'.join(dna_sequence_variants_df['Gene_Coding'].astype(str)) if not dna_sequence_variants_df.empty else 'NOT IDENTIFIED'
    else: 
        gene_mutations = 'NOT IDENTIFIED'
    
    # Process copy number variations DataFrame
    if not copy_number_variations_merged_with_filter_df.empty:
        copy_number_variants_df = copy_number_variations_merged_with_filter_df.copy()
        copy_number_variants_df['Gene_Coding']=copy_number_variants_df['Gene']+ ' (' + copy_number_variants_df['CytoBand']+ ')'
        # Concatenate copy number variants into a string separated by newline characters
        copy_number_variants = '\n'.join(copy_number_variants_df['Gene_Coding'].astype(str)) if not copy_number_variants_df.empty else 'NOT IDENTIFIED'
    else: 
        copy_number_variants = 'NOT IDENTIFIED'

    # Process fusion variants DataFrame
    if not fusion_variants_df.empty:
        gene_fusions_df = fusion_merged_with_filter_df.copy()
        # Concatenate gene fusions into a string separated by newline characters
        gene_fusions = '\n'.join(gene_fusions_df['Genes (Exons)'].astype(str)) if not gene_fusions_df.empty else 'NOT IDENTIFIED'
    else: 
        gene_fusions = 'NOT IDENTIFIED'
    
    # Prepare data for the summary table
    table_data = {
        "Gene Mutations": gene_mutations,
        "Gene Fusions": gene_fusions,  
        "Copy number variants (CNVs)": copy_number_variants,
        "Loss of Heterozygosity (LOH)": loh_score_format,
        "Tumor mutational burden (TMB)": tmb_status_score,
        "Microsatellite instability (MSI)": msi_value,}
    
    # Generate the summary table in the document
    report_summary_table(document, table_data)
    
    # Formatting for notes at the end of the document
    document.paragraphs[-1].paragraph_format.space_before = Pt(0)
    
    # Add first note
    note1 = document.add_paragraph("*Note:This High TMB Score might be affected by higher deamination in this sample which may be due  to  the intrinsic nature of FFPE tissue.")
    note1.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note1.paragraph_format.left_indent = Cm(2.5)
    note1.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    note1.paragraph_format.line_spacing = 1
    for run in note1.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
    
    # Add second note
    note2 = document.add_paragraph("**Note: Assessment of HRR/HRD using orthogonal methods is recommended.")
    note2.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note2.paragraph_format.left_indent = Cm(2.5)
    note2.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    for run in note2.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'

In [15]:
#Creates an empty therapeutic table 
def therapeutic_summary_table(document):
    print('Adding Therapeutic summary table')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
        
    line1_paragraph = document.add_paragraph("Therapeutic summary is relevant for the patient’s diagnosis")
    line1_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line1_paragraph.paragraph_format.left_indent = Cm(2.5)
    line1_paragraph.paragraph_format.right_indent = Cm(2.5)
    line1_paragraph.paragraph_format.space_after = Cm(0)
    line2_paragraph = document.add_paragraph("(Additional in-trial therapies relevant for this patient are listed in the 'clinical trials' section)")
    line2_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line2_paragraph.paragraph_format.left_indent = Cm(2.5)
    line2_paragraph.paragraph_format.right_indent = Cm(2.5)
    line2_paragraph.paragraph_format.space_after = Cm(0)
    for run in line1_paragraph.runs + line2_paragraph.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    paragraph = document.add_paragraph() 
    table = document.add_table(rows=10, cols=6)  
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(1.5)
    table.columns[1].width = Cm(2.5)
    table.columns[2].width = Cm(1.5)
    table.columns[3].width = Cm(5.96)
    table.columns[4].width = Cm(4)
    table.columns[5].width = Cm(3.75)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style = 'Table Grid'

    header = table.rows[0].cells
    header[0].text = "Tier"
    header[1].text = "Variant/ Biomarker"
    header[2].text = "VAF"
    header[3].paragraphs[0].add_run("Therapy (Approved/ In-trial#) \n (Please see details in the “Approved therapy” and “Clinical trials” sections of the report)")
    header[4].paragraphs[0].add_run("Potential drug resistance \n (Please see details in “Drug resistance” sections)")
    header[5].paragraphs[0].add_run("Prognostic \n (Please see details in “Prognostic significance” sections)")

    for cell in header:
        cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].bold = True

    for row in table.rows[1:]:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(10)
                    run.font.name = 'Times New Roman'   
    return table 

In [16]:
def therapeutic_summary(document):
    print('In Therapeutic Summary section')
    # Set margins for the document
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    
    # Add image to the document
    new_width = Cm(12.95)
    new_height = Cm(0.57)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Therapeutic_summary.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    
    # Generate therapeutic summary table
    therapeutic_summary_table(document)
    
    # Format notes at the end of the document
    document.paragraphs[-1].paragraph_format.space_before = Pt(0)
    
    # Add Note 1
    note1 = document.add_paragraph("Note 1: The therapies in bold have been approved by the FDA/NCCN. Further information about these therapies can be found in the “Immunotherapy” and “Approved therapy” section of this report.")
    note1.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note1.paragraph_format.left_indent = Cm(2.5)
    note1.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    note1.paragraph_format.line_spacing = 1
    for run in note1.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    
    # Add Note 2
    note2 = document.add_paragraph("Note 2: The therapies labeled as # are being tested in registered clinical trials for which the patient may be eligible to enroll. Please see the “Clinical trials” section for more details.")
    note2.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note2.paragraph_format.left_indent = Cm(2.5)
    note2.paragraph_format.right_indent = Cm(2.5)
    note2.paragraph_format.space_after = Cm(0)
    for run in note2.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    
    # Add Note 3
    note3 = document.add_paragraph("Note 3: The *Drug resistance associations mentioned in the above table may be observed in preclinical or clinical studies. More details can be found in the “Drug resistance” section for each variation.")
    note3.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note3.paragraph_format.left_indent = Cm(2.5)
    note3.paragraph_format.right_indent = Cm(2.5)
    note3.paragraph_format.space_after = Cm(0)
    for run in note3.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    
    # Add Note 4
    note4 = document.add_paragraph("Note 4: Prognostic significance of Tier II and other mutations may be contextual and may vary in different studies. This section indicates the published data relevant for the variant and patient’s tumor with respect to Therapy response, Metastasis, Overall survival (OS), Progression free survival (PFS), Recurrence free survival (RFS), Histological subtype, Germline risk etc.")
    note4.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note4.paragraph_format.left_indent = Cm(2.5)
    note4.paragraph_format.right_indent = Cm(2.5)
    note4.paragraph_format.space_after = Cm(0)
    for run in note4.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'

    # Add Note 5
    note5 = document.add_paragraph("Note 5: The “Pathogenic”, “Likely pathogenic” and “VUS-deleterious” mutations are interpreted and Tier wise prioritization is done as per AMP/ASCO/ACMG  joint consensus.")
    note5.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note5.paragraph_format.left_indent = Cm(2.5)
    note5.paragraph_format.right_indent = Cm(2.5)
    note5.paragraph_format.space_after = Cm(0)
    for run in note5.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'

In [17]:
def add_formatted_paragraph(doc, text):
    # Add a formatted paragraph with specified text to the document
    paragraph = doc.add_paragraph()
    run = paragraph.add_run(text)
    run.font.name = 'Times New Roman'
    run.font.size = Pt(12)
    paragraph.paragraph_format.line_spacing = 1.0
    paragraph.paragraph_format.left_indent = Cm(2.5)
    paragraph.paragraph_format.right_indent = Cm(2.5)
    return paragraph

def immunotherapy_markers(document):
    print('In Immunotherapy associated markers section')
    global tmb_status_score, msi_value, mmr_status, patient_info
    '''
    Immunotherapy marker section information includes PD-L1 status, tumor mutation burden, 
    microsatellite instability,and mismatch repair status, along with an image header and specific paragraph and margin styles.
    '''
    # Set margins for the document
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    
    # Add image to the document
    new_width = Cm(10.27)
    new_height = Cm(0.66)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Immunotherapy_markers.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    
    # Get PD-L1 status information
    pdl1_status = patient_info.get('PD-L1_status', '') or ''
    pdl1_status_score = patient_info.get('PD-L1_status_score', '') or ''
    combined_pdl1_value = f"{pdl1_status}({pdl1_status_score})" if pdl1_status_score else pdl1_status  
    
    # Define MMR genes and a function to check for them
    mrr_genes = ['MSH2', 'MSH6', 'MLH1', 'PMS2'] 
    def check_MRR_genes(df, genes):
        found_genes = []
        for gene in genes:
            if gene in df['Gene'].values:
                found_genes.append(gene)
        return found_genes
            
    # Check MMR genes in DNA sequence variants and copy number variations dataframes
    if 'Gene' in dna_sequence_variants_df.columns:
        found_genes_dna = check_MRR_genes(dna_sequence_variants_df, mrr_genes)
    else:
        found_genes_dna = []
    
    if not copy_number_variations_df.empty:
        if 'Gene' in copy_number_variations_df.columns:
            found_genes_copy = check_MRR_genes(copy_number_variations_df, mrr_genes)
    else:
        found_genes_copy = []
    
    # Combine found MMR genes from both dataframes
    found_genes = found_genes_dna + found_genes_copy
    if found_genes:
        mmr_status = "Loss-of-function mutation was found in the MMR gene/s: " + ", ".join(found_genes) + '\n' + "IHC is recommended to confirm the nuclear loss of MMR proteins"
    else:
        mmr_status = f"No loss-of-function mutation was found in the 4 MMR genes; {', '.join(mrr_genes)}"

    # Add formatted paragraphs for immunotherapy markers
    add_formatted_paragraph(document, f"1- PD-L1 by IHC (SP263/22C3): {combined_pdl1_value}")
    add_formatted_paragraph(document, f"2- Tumor Mutation Burden: {tmb_status_score}")
    add_formatted_paragraph(document, f"3- Microsatellite Instability: {msi_value}")
    add_formatted_paragraph(document, f"4- MMR (Mismatch repair) status: {mmr_status}")

In [18]:
# Function to add a hyperlink to a paragraph in a Word document
def add_hyperlink(paragraph, text, url):
    # Get the part of the paragraph to relate to the hyperlink
    part = paragraph.part
    # Create a relationship ID for the hyperlink
    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
    # Create a new hyperlink element
    hyperlink = docx.oxml.shared.OxmlElement('w:hyperlink')
    hyperlink.set(docx.oxml.shared.qn('r:id'), r_id)
    # Create a new run for the hyperlink text
    new_run = docx.text.run.Run(docx.oxml.shared.OxmlElement('w:r'), paragraph)
    new_run.text = text
    # Set the hyperlink style
    new_run.style = get_or_create_hyperlink_style(part.document)
    # Append the run to the hyperlink element
    hyperlink.append(new_run._element)
    # Append the hyperlink to the paragraph
    paragraph._p.append(hyperlink)
    return hyperlink

# Function to create or retrieve the hyperlink style in the document
def get_or_create_hyperlink_style(document):
    if "Hyperlink" not in document.styles:
        if "Default Character Font" not in document.styles:
            default_character_font = document.styles.add_style("Default Character Font", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
            default_character_font.element.set(docx.oxml.shared.qn('w:default'), "1")
            default_character_font.priority = 1
            default_character_font.hidden = True
            default_character_font.unhide_when_used = True
            del default_character_font
        hyperlink_style = document.styles.add_style("Hyperlink", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
        hyperlink_style.base_style = document.styles["Default Character Font"]
        hyperlink_style.unhide_when_used = True
        hyperlink_style.font.color.rgb = docx.shared.RGBColor(0x05, 0x63, 0xC1)
        hyperlink_style.font.underline = True
        del hyperlink_style
    return "Hyperlink"

# Main function to create a section in the document with approved immunotherapy information
def approved_immunotherapy(document):
    print('In Approved Immunotherapy section')
    global patient_info, mutational_burden, mmr_status, msi_value
    
    # Set the margins for each section in the document
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
        
    new_width = Cm(9.58)
    new_height = Cm(0.69)
    # Add a paragraph and insert an image (heading)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Approved_immunotherapy.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)

    # Extract main tumor type from Redmine
    main_tumor_types = patient_info.get('Main Tumor Type', '')
    
    # Process the mutational burden value
    if mutational_burden is not None:
        numeric_part = ''.join(c for c in mutational_burden if c.isdigit() or c == '.')
    try:
        if numeric_part:
            mutational_burden_value = float(numeric_part)
            mutational_burden_status = 'High' if mutational_burden_value >= TMB_CUTOFF else 'Low'
        else:
            mutational_burden_status = 'Unknown'
    except ValueError:
        mutational_burden_status = 'Unknown'
    
    # Determine MSI status
    if 'instability' in msi_value.lower():
        msi_statuss = 'MSI-High'
    else:
        msi_statuss = 'MSS'
        
    # Determine PD-L1 status
    pdl1_status_score = patient_info.get('PD-L1_status_score', '') or ''
    if pdl1_status_score is None or pdl1_status_score == 'Not requested':
        pdl1_status = 'Unknown'
    elif 'Positive' in pdl1_status_score:
        pdl1_status = 'Positive'
    else:
        pdl1_status = 'Negative'
        
    # Determine MMR status result
    if mmr_status == 'No loss-of-function mutation in the 4 MMR genes: MSH2, MSH6, MLH1, PMS2':
        mmr_status_result = 'likely MMR proficient'
    else:
        mmr_status_result = 'likely MMR deficient'
        
    # Create a summary paragraph with the determined statuses
    p = document.add_paragraph()        
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run(f"This is a PD-L1 {pdl1_status}, ")
    p.add_run(f"TMB-{mutational_burden_status}, ")
    p.add_run(f"{msi_statuss}, ")
    p.add_run(f"{mmr_status_result} case of {main_tumor_types}")    
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    
    # Split the main tumor types and read the FDA approved immunotherapies from an Excel file
    tumor_types = patient_info.get('Main Tumor Type', '').split(', ')
    fda_approved_df = pd.read_excel(DB_PATH + IMMUNOTHERAPIES_DB_FILE)
    filtered_df = fda_approved_df[fda_approved_df['Main tumor type'].str.contains('Solid tumours', case=False)]
    
    # Initialize a flag to track if any approved therapies are found
    therapies_found = False

    # Loop through each tumor type and find matching immunotherapies
    for tumor_type in tumor_types:
        matching_rows2 = fda_approved_df[fda_approved_df['Main tumor type'].str.contains(fr'{tumor_type}', case=False, regex=True, na=False, flags=re.IGNORECASE)]        
        if not matching_rows2.empty:
            therapies_found = True
        for index, row in matching_rows2.iterrows():
            drug_name = row['Immunotherapy']
            link_holder = document.add_paragraph()
            link_holder.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            link_holder.paragraph_format.left_indent = Cm(2.5)
            link_holder.paragraph_format.right_indent = Cm(2.5)
            drug = drug_name.replace(" ", "+")
            add_hyperlink(link_holder, drug_name, f"https://www.google.com/search?q={drug}+mechanism+of+action")
            
            main_tumor_type = row['Main tumor type']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(main_tumor_type)
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name = 'Times New Roman'
            
            detailed_tumor_type = row['FDA-approved indications']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(detailed_tumor_type)
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name = 'Times New Roman'
            
            biomarker_details = row['Biomarker']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(str(biomarker_details))
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name = 'Times New Roman'
            
            details_of_approval = row['Details of approval']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(details_of_approval)
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name = 'Times New Roman'

    matching_rows3 = pd.DataFrame()
    
    # Find additional matching rows based on biomarker details
    if mmr_status_result == 'likely MMR deficient':
        matching_rows3 = pd.concat([matching_rows3, filtered_df[filtered_df['Biomarker'].str.contains('dMMR')]])

    if msi_statuss == 'MSI-High':
        matching_rows3 = pd.concat([matching_rows3, filtered_df[filtered_df['Biomarker'].str.contains('MSI-High')]])
    
    if mutational_burden_status == 'High':
        matching_rows3 = pd.concat([matching_rows3, filtered_df[filtered_df['Biomarker'].str.contains('TMB-High')]])
    
    matching_rows3.drop_duplicates(inplace=True)
    
    # Add the additional matching rows to the document
    for index, row in matching_rows3.iterrows():
        therapies_found = True
        drug_name = row['Immunotherapy']
        link_holder = document.add_paragraph()
        link_holder.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        link_holder.paragraph_format.left_indent = Cm(2.5)
        link_holder.paragraph_format.right_indent = Cm(2.5)
        drug = drug_name.replace(" ", "+")
        add_hyperlink(link_holder, drug_name, f"https://www.google.com/search?q={drug}+mechanism+of+action")
        
        main_tumor_type = row['Main tumor type']
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(main_tumor_type)
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name = 'Times New Roman'
        
        detailed_tumor_type = row['FDA-approved indications']
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(detailed_tumor_type)
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name = 'Times New Roman'
        
        biomarker_details = row['Biomarker']
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(biomarker_details)
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name = 'Times New Roman'
        
        details_of_approval = row['Details of approval']
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(details_of_approval)
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name = 'Times New Roman'
    
    # If no therapies were found, add a paragraph indicating this
    if not therapies_found:
        no_therapy_paragraph = document.add_paragraph()
        no_therapy_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        no_therapy_paragraph.paragraph_format.left_indent = Cm(2.5)
        no_therapy_paragraph.paragraph_format.right_indent = Cm(2.5)
        no_therapy_paragraph.add_run("No approved therapy was found in the in-house database.")
        for run in no_therapy_paragraph.runs:
            run.font.size = Pt(12)
            run.font.name = 'Times New Roman'

In [19]:
# Function to create a section in the document with information about immunotherapies in clinical trials
def intrial_immunotherapy(document):
    print('Intrial immunotherapy section')
    # Set the margins for each section in the document
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    
    # Define the dimensions for the heading image
    new_width = Cm(9.58)
    new_height = Cm(0.56)
    
    # Add a paragraph and insert the heading image
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0  
    image_path = os.path.join("Heading_Images", 'Intrial_immunotherapy.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    
    # Add a paragraph with information about immunotherapies in clinical trials
    paragraph = document.add_paragraph("Multiple immunotherapies are being investigated in clinical trials for this patient’s genomic profile and/or diagnosis (Please see “Clinical trials” section for more details).")
    paragraph.paragraph_format.left_indent = Cm(2.5)  
    paragraph.paragraph_format.right_indent = Cm(2.5)  
    
    # Set the font and size for the added text
    run_text = paragraph.runs[0]
    font = run_text.font
    font.name = 'Times New Roman'
    font.size = Pt(12)

In [20]:
# Function to process DNA sequence variants and create the SNV dataframe
def process_dna_sequence_variants():
    global snv_df, dna_sequence_variants_merged_with_filter_df
    
    if dna_sequence_variants_merged_with_filter_df.empty:
        print("The dna_sequence_variants_merged_with_filter_df DataFrame is empty.")
        snv_df= None
    else: 
        try:
            dna_sequence_variants_df['Genes'] = dna_sequence_variants_merged_with_filter_df['Gene'] + '\n' + dna_sequence_variants_merged_with_filter_df['Transcript']
            selected_columns_dsv = dna_sequence_variants_df.loc[:, ['Genes', 'Genomic alterations', 'Allele Frequency', 'Oncomine Gene Class', 'Classification', 'Tier']]
            #print(selected_columns_dsv )
            selected_columns_dsv.columns = ['Gene', 'Genomic alteration', 'Variant Allelic Frequency (VAF%)', 'Impact on Protein Function', 'Clinical Significance', 'AMP^ Classification']
            snv_df= selected_columns_dsv.copy()
        except NameError:
            snv_df= None

# Function to process copy number variants and create the CNV dataframe
def process_copy_number_variants():
    global cnv_df, copy_number_variations_merged_with_filter_df
    if copy_number_variations_merged_with_filter_df.empty:
        print("The copy_number_variations_merged_with_filter_df DataFrame is empty.")
        cnv_df= None
    else: 
        try:
            selected_columns_cnv = copy_number_variations_merged_with_filter_df.loc[:, ['Gene', 'Locus', 'Oncomine Variant Class', 'Oncomine Gene Class', 'Copy Number', 'Length']]
            selected_columns_cnv.columns = ['Gene', 'Locus', 'CNV Type', 'Impact on Protein Function', 'Copy number', 'CNV size (bp)']
            cnv_df= selected_columns_cnv.copy()
        except NameError:
            cnv_df= None

# Function to process fusion variants and create the fusions dataframe
def process_fusion_variants():
    global fusions_df, fusion_merged_with_filter_df
    if fusion_merged_with_filter_df.empty:
        print("The fusion_merged_with_filter_df DataFrame is empty.")
        fusions_df= None
    else:
        try:
            selected_columns_fusion = fusion_merged_with_filter_df.loc[:,['Genes', 'Locus', 'Read Count', 'Oncomine Gene Class', 'Oncomine Variant Class', 'Tier']]
            selected_columns_fusion.columns = ['Gene', 'Locus', 'Coverage/VAF', 'Impact on protein Function', 'Variant type', 'Tier Classification']
            fusions_df= selected_columns_fusion.copy()
        except NameError:
            fusions_df= None

# Function to add tables of variant details to the document
def variant_detail_table1(document, snv_df, fusions_df, cnv_df):
    print('Adding Table 1 in Variant description section')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(1.5)
        section.right_margin = Cm(2)
    
    tables = [] 

    # SNV Table
    if snv_df is not None and not snv_df.empty:
        table = document.add_table(rows=snv_df.shape[0] + 1, cols=snv_df.shape[1])
        table.style = 'Table Grid'
        table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for col_num, col_name in enumerate(snv_df.columns):
            cell = table.cell(0, col_num)
            cell.text = col_name
            run = cell.paragraphs[0].runs[0]
            run.bold = True
            run.font.name = 'Times New Roman'
            run.font.size = Pt(10)
            cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for row_num in range(snv_df.shape[0]):
            for col_num, value in enumerate(snv_df.iloc[row_num]):
                cell = table.cell(row_num + 1, col_num)
                cell.text = str(value)
                run = cell.paragraphs[0].runs[0]
                run.font.name = 'Times New Roman'
                run.font.size = Pt(10)
                cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        if snv_df.shape[0] < len(table.rows):
            last_row = table.rows[-1]
            if all(cell.text == '' for cell in last_row.cells):
                table._element.remove(last_row._element)
        tables.append(table)  
    
    # Fusion Table
    if fusions_df is not None and not fusions_df.empty:
        table = document.add_table(rows=fusions_df.shape[0] + 1, cols=fusions_df.shape[1])
        table.style = 'Table Grid'
        table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for col_num, col_name in enumerate(fusions_df.columns):
            cell = table.cell(0, col_num)
            cell.text = col_name
            run = cell.paragraphs[0].runs[0]
            run.bold = True
            run.font.name = 'Times New Roman'
            run.font.size = Pt(10)
            cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for row_num in range(fusions_df.shape[0]):
            for col_num, value in enumerate(fusions_df.iloc[row_num]):
                cell = table.cell(row_num + 1, col_num)
                cell.text = str(value)
                run = cell.paragraphs[0].runs[0]
                run.font.name = 'Times New Roman'
                run.font.size = Pt(10)
                cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        if fusions_df.shape[0] < len(table.rows):
            last_row = table.rows[-1]
            if all(cell.text == '' for cell in last_row.cells):
                table._element.remove(last_row._element)
        tables.append(table)  

    # CNV Table
    if cnv_df is not None and not cnv_df.empty:
        table = document.add_table(rows=cnv_df.shape[0] + 1, cols=cnv_df.shape[1])
        table.style = 'Table Grid'
        table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for col_num, col_name in enumerate(cnv_df.columns):
            cell = table.cell(0, col_num)
            cell.text = col_name
            run = cell.paragraphs[0].runs[0]
            run.bold = True
            run.font.name = 'Times New Roman'
            run.font.size = Pt(10)
            cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for row_num in range(cnv_df.shape[0]):
            for col_num, value in enumerate(cnv_df.iloc[row_num]):
                cell = table.cell(row_num + 1, col_num)
                cell.text = str(value)
                run = cell.paragraphs[0].runs[0]
                run.font.name = 'Times New Roman'
                run.font.size = Pt(10)
                cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        if cnv_df.shape[0] < len(table.rows):
            last_row = table.rows[-1]
            if all(cell.text == '' for cell in last_row.cells):
                table._element.remove(last_row._element)
        tables.append(table)  
    
    return tables 

In [21]:
def variant_detail_table2(document):
    print('Adding Table 2 in Variant details section')
    global dna_sequence_variants_merged_with_filter_df
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(1.5)
        section.right_margin = Cm(2)
    
    table = document.add_table(rows=4, cols=8)  
    table.autofit = False
    table.style = 'Table Grid'
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    
    widths = [2.5, 2.5, 2.5, 2.6, 1.25, 1.25, 1.25, 1.75, 2.5]  
    for i, width in enumerate(widths):
        table.cell(0, i).width = Cm(width)

    heights = [1, 1, 1, 1]
    for i, height in enumerate(heights):
        table.rows[i].height = Cm(height)
        
    header = table.rows[0].cells
    header[0].text = "Variant"
    table.cell(0, 0).merge(table.cell(1, 0))
    header[1].text = "dbSNP\nrs ID"
    table.cell(0, 1).merge(table.cell(1, 1))
    header[2].text = "COSMIC"
    table.cell(0, 2).merge(table.cell(1, 2))
    header[3].text = "ClinVar"
    table.cell(0, 3).merge(table.cell(1, 3))
    header[4].text = "In-silico predictors"
    table.cell(0, 4).merge(table.cell(0, 6))
    header[7].text = "Population Database"
    table.cell(0, 7).merge(table.cell(0, 7))
    
    second_row = table.rows[1].cells
    second_row[0].text = "Variant"
    second_row[1].text = "dbSNP\nrs ID"
    second_row[2].text = "COSMIC"
    second_row[3].text = "ClinVar"
    second_row[4].text = "SIFT"
    second_row[5].text = "Polyphen"
    second_row[6].text = "MT2"
    second_row[7].text = "gnomAD"
    
    header = table.rows[0].cells
    second_row = table.rows[1].cells
    for i in range(min(len(header), len(second_row), 9)):
        header[i].paragraphs[0].runs[0].font.bold = True
        header[i].paragraphs[0].runs[0].font.name = 'Times New Roman'
        header[i].paragraphs[0].runs[0].font.size = Pt(11)
        header[i].paragraphs[0].paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        header[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER

        second_row[i].paragraphs[0].runs[0].font.bold = True
        second_row[i].paragraphs[0].runs[0].font.name = 'Times New Roman'
        second_row[i].paragraphs[0].runs[0].font.size = Pt(10)
        second_row[i].paragraphs[0].paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        second_row[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER

    try:
        total_rows_needed = len(dna_sequence_variants_merged_with_filter_df) + 2 
        while len(table.rows) < total_rows_needed:
            table.add_row()
        dna_sequence_variants_merged_with_filter_df = dna_sequence_variants_merged_with_filter_df.fillna('').astype(str)
        for i, (index, data) in enumerate(dna_sequence_variants_merged_with_filter_df.iterrows()):

            row = table.rows[i + 2]
            row.cells[0].text = data.get("Genes_codings", "")
            row.cells[1].text = data.get("dbSNP", "")
            variant_id = data.get("Variant ID", "")
            clinvar_id = data.get("Clinvar_accession_no", "")
            # If the "Variant ID" starts with "COSM", it's assigned to the third cell. If not, "-" is assigned
            if variant_id:
                if variant_id.startswith("COSM"):
                    row.cells[2].text = variant_id
            # If "Clinvar_accession_no" exists, its value is assigned to the fourth cell; otherwise, "-" is assigned.
            else:
                row.cells[2].text = "-" 
            if clinvar_id:
                row.cells[3].text = str(clinvar_id)
            else:
                row.cells[3].text = "-"  
                
            # If a value exists, it's converted to float, and based on a threshold (SIFT_CUTOFF), it's classified as "D" or "B" and assigned to the fifth cell. If no value exists, "-" is assigned.
            sift_value = data.get("SIFT", "")
            if sift_value:
                sift_score = float(sift_value)
                if sift_score <= SIFT_CUTOFF:
                    sift_classification = "D"
                else:
                    sift_classification = "B"
            else:
                sift_classification = "-"
            row.cells[4].text = sift_classification

            # If a value exists, it's converted to float, Based on threshold (POLYPHEN_MIN, POLYPHEN_MAX) , it's classified as "B", "PD", or "D" and assigned to the sixth cell. If no value exists, "-" is assigned.
            polyphen_value = data.get("PolyPhen", "")
            if polyphen_value:
                polyphen_score = float(polyphen_value)
                if polyphen_score <= POLYPHEN_MIN:
                    polyphen_classification = "B"
                elif POLYPHEN_MIN < sift_score < POLYPHEN_MAX:
                    polyphen_classification = "PD"
                else:
                    polyphen_classification = "D"
            else:
                polyphen_classification = "-"
            row.cells[5].text = polyphen_classification
    
            row.cells[6].text = data.get("MT2", "")
            gnomAD = data.get("gnomAD_value", "")
            row.cells[7].text = str(gnomAD)

            
        for row in table.rows[2:]:
            for cell in row.cells:
                cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
                for paragraph in cell.paragraphs:
                    if paragraph.runs and paragraph.runs[0].text.strip():
                        paragraph.runs[0].font.size = Pt(9.5)
                        paragraph.runs[0].font.name = 'Times New Roman'
                        paragraph.paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    
        return table
    except NameError:
        print("dna_sequence_variants_df does not exist.")

In [22]:
def variant_details(document):
    print('In Variant Details section')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(9.58)
    new_height = Cm(0.56)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Variant_details.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Table-1: List of variants identified in this sample").bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    # Process the dataframes
    process_dna_sequence_variants()
    process_fusion_variants()
    process_copy_number_variants()
    variant_detail_table1(document, snv_df, fusions_df, cnv_df)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Refer to supplementary information for the classification criteria details. ^")
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Table-2: Significance of Variants reported in database").bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    variant_detail_table2(document)
    
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("PD- Possibly Damaging, D- Damaging/Deleterious, B-Benign/Tolerated")
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()

In [23]:
def process_dna_altered_df():
    global dna_sequence_variants_merged_with_filter_df
    #print(dna_sequence_variants_merged_with_filter_df)
    document = Document()
    table = variant_detail_table2(document)
    # Extracting data from a table
    dna_altered_df = pd.DataFrame([[cell.text.strip() for cell in row.cells] for row in table.rows[2:]],
                                  columns=["Variant", "dbSNP rs ID", "COSMIC", "ClinVar", 
                                           "SIFT", "Polyphen", "MT2", "gnomAD"])
    # Renaming column for consistency
    dna_altered_df.rename(columns={'Variant': 'Genes_codings'}, inplace=True)
    #print(dna_altered_df)
    try: 
        # Copying a DataFrame to avoid modification of original data
        dna_sequence_variants_df1 = dna_sequence_variants_merged_with_filter_df.copy()
        dna_sequence_variants_df1.rename(columns={'ClinVar': 'Significance'}, inplace=True)
        # Merging two DataFrames based on a common column
        merged_df = pd.merge(dna_altered_df, dna_sequence_variants_df1, on="Genes_codings", how="inner")
        #print(merged_df.columns)
        #display(merged_df['Coverage'])
        
        # Data processing and mapping
        sift_mapping = {'D': 'damaging/deleterious', 'B': 'benign/tolerated', 'PD': 'possibly damaging'}
        polyphen_mapping = {'D': 'damaging/deleterious', 'B': 'benign', 'PD': 'possibly damaging'}
        mt2_mapping = {'D': 'deleterious', 'B': 'benign'}
        paragraphs_list = []
        
        #print(merged_df.columns)
        for index, row in merged_df.iterrows():
            # Extracting and formatting data for each row
            locus_parts = row['Locus'].split(':') if row['Locus'] else ('', '')
            chromosome, position = locus_parts[0].replace('chr', '').replace('Chr', ''), locus_parts[1] if len(locus_parts) > 1 else ''
            gene = row['Gene'] if row['Gene'] else '________'
            coding = row['Coding'] if row['Coding'] else '________'
            amino_acid_change = row['Amino Acid Change'] if row['Amino Acid Change'] else '________'
            allele_frequency = row['Allele Frequency'] if row['Allele Frequency'] else '________'
            allele_coverage_alt = row['Allele Coverage Alt'] if not pd.isna(row['Allele Coverage Alt']) else '________'
            coverage = row['Coverage'] if row['Coverage'] else '________'
            exon = row['Exon'] if row['Exon'] else '________'
            transcript = row['Transcript'] if row['Transcript'] else '________'
            dbSNP_rs_ID = row['dbSNP rs ID'] if row['dbSNP rs ID'] else '________'
            cosmic = row['COSMIC'] if row['COSMIC'] else '________'
            significance = row['Classification'] if row['Classification'] else '________'
            clinvar = row['Clinvar_accession_no'] if row['Clinvar_accession_no'] else 'unknown'
            sift_prediction = row['SIFT_x']
            sift_classification = sift_mapping.get(sift_prediction, 'unknown') if sift_prediction != '-' else 'unknown'
            polyphen_prediction = row['Polyphen']
            polyphen_classification = polyphen_mapping.get(polyphen_prediction, 'unknown') if polyphen_prediction != '-' else 'unknown'
            mt2_prediction = row['MT2_x']
            mt2_classification = mt2_mapping.get(mt2_prediction, 'unknown') if mt2_prediction != '-' else 'unknown'
            gnomAD = row['gnomAD']if row['gnomAD']else '-'
            gen_pro = row['1000_Genomes_value'] if row['1000_Genomes_value']else '-'
            
            # Generating paragraphs with formatted data
            paragraph = f"{significance} \n"\
                        f"The {amino_acid_change} variant (also known as {coding}), " \
                        f"was detected in {gene} gene on chromosome {chromosome} " \
                        f"at position {position} with a variant allele frequency of {allele_frequency} " \
                        f"(represented by {allele_coverage_alt} reads). " \
                        f"This mutation has a total depth of {coverage}. " \
                        f"It is located at exon {exon} of {transcript} transcript " \
                        f"and was found to change amino acid {amino_acid_change}. " \
                        f"It is represented by {dbSNP_rs_ID} in dbSNP and {cosmic} " \
                        f"in the Cosmic database. It is interpreted as {significance.lower()} according to the ClinVar [{clinvar}] " \
                        f"database. It is predicted as {sift_classification} by SIFT, {polyphen_classification} " \
                        f"by PolyPhen2 and {mt2_classification} by MutationTaster2, which are in-silico DNA variant effect prediction tools. " \
                        f"In the population frequency databases, this variant is present, with a global minor allele frequency of {gnomAD}% in gnomAD "\
                        f"and {gen_pro}% in 1000G database."
        
            paragraphs_list.append(paragraph)
            #print(paragraphs_list)
        # Adding newly generated paragraphs to the DataFrame
        dna_sequence_variants_merged_with_filter_df['altered_variant_para'] = paragraphs_list
        # Creating a new column by combining two existing columns
        dna_sequence_variants_merged_with_filter_df['OncoKB_input'] = dna_sequence_variants_merged_with_filter_df['Gene'] + ' ' + dna_sequence_variants_merged_with_filter_df['Amino Acid Change']
        #print(dna_sequence_variants_merged_with_filter_df)
    except :
        print("DataFrame 'dna_sequence_variants_df' is not available.")


# Function to construct paragraph for gene fusions
def construct_paragraph(row):
    fusion = row['Genes (Exons)']
    gene = row['Gene']
    exon = row['Exon']
    reads = row['Read Count']
    ogc = row['Oncomine Gene Class']
    locus = row['Locus']

    exon_numbers = re.findall(r'\((\d+)\)', row['Genes (Exons)'])
    
    # Constructing paragraph
    paragraph = (
        f'This fusion was detected between exon {exon_numbers[0]} of gene {gene.split(", ")[0]} '
        f'and exon {exon_numbers[1]} of gene {gene.split(", ")[1]} located at genomic position '
        f'{locus.split(" - ")[0].split(":")[1]} of chromosome {locus.split(" - ")[0].split(":")[0]} '
        f'and position {locus.split(" - ")[1].split(":")[1]} of chromosome {locus.split(" - ")[1].split(":")[0]} '
        f'respectively. This fusion is supported by {reads} reads. '
        f'This fusion is represented as {fusion}. It results in {ogc}.'
    )
    return paragraph

# Function to process gene fusion DataFrame
def process_gene_fusions_df():
    global fusion_merged_with_filter_df
    
    try:
        # Check if the DataFrame is empty or None
        if fusion_merged_with_filter_df is None or fusion_merged_with_filter_df.empty:
            print("DataFrame 'fusion_merged_with_filter_df' is empty")
            return

        # Copy DataFrame to avoid modifying original data
        gene_fusions_df1 = fusion_merged_with_filter_df.copy()
        #print(gene_fusions_df1)
        
        # Prepare 'Gene' column by replacing '-' and '::' with ', ' 
        gene_fusions_df1['Gene'] = gene_fusions_df1['Genes'].apply(lambda x: x.replace('-', ', ').replace('::', ', '))
        
        # Assuming the necessary columns are present, construct paragraphs
        gene_fusions_df1['Exon'] = gene_fusions_df1['Genes (Exons)'].str.findall(r'\((\d+)\)').apply(lambda x: ', '.join(x))
        fusion_merged_with_filter_df['altered_variant_para'] = gene_fusions_df1.apply(construct_paragraph, axis=1)
        
    except:
        print("DataFrame 'fusion_merged_with_filter_df' is not available.")


def process_copy_number_variants_df():
    global copy_number_variations_merged_with_filter_df
    
    try:
        # Check if the DataFrame is empty or None
        if copy_number_variations_merged_with_filter_df is None or copy_number_variations_merged_with_filter_df.empty:
            print("DataFrame 'copy_number_variations_merged_with_filter_df' is empty")
            return
        
        # Check if the necessary columns exist in the DataFrame
        if 'Gene' not in copy_number_variants_df1.columns or 'CytoBand' not in copy_number_variants_df1.columns:
            print("Required columns 'Gene' or 'CytoBand' not found in the DataFrame.")
            return
            
        # Copy the DataFrame to avoid modifying the original data
        copy_number_variants_df1 = copy_number_variations_merged_with_filter_df.copy()
        # Create a new column combining 'Gene' and 'CytoBand'
        copy_number_variants_df1['Gene_Coding'] = copy_number_variants_df1['Gene'] + ' (' + copy_number_variants_df1['CytoBand'] + ')'
        
        # Initialize a list to store constructed paragraphs
        paragraphs = []
        for index, row in copy_number_variants_df1.iterrows():
            # Extract necessary information from the row
            gene = row['Gene']
            cn = row['Copy Number']
            ogc = row['Oncomine Gene Class']
            coding = row['Gene_Coding']
            ovc = row['Oncomine Variant Class']
            length = row['Length']
            chromosome, position = row['Locus'].split(':')[0], row['Locus'].split(':')[1]
            
            # Construct the paragraph
            paragraph = (
                f"{ovc} type of variant was detected in {gene} gene on chromosome {chromosome} from genomic position {position}. "
                f"This variant is of length {length} bp, having copy number gain of {cn} copies. It results in {ogc}.")
            
            # Append the constructed paragraph to the list
            paragraphs.append(paragraph)
        
        # Add the list of paragraphs to a new column in the DataFrame
        copy_number_variations_merged_with_filter_df['altered_variant_para'] = paragraphs
            
    except:
        print("DataFrame 'copy_number_variants_df' is not available.")

In [24]:
# Function to add a hyperlink to a paragraph in a Word document
def add_hyperlink(paragraph, drug_name, url):
    # Get the part of the paragraph
    part = paragraph.part
    # Create a relationship ID for the hyperlink
    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
    # Create a hyperlink element
    hyperlink = docx.oxml.shared.OxmlElement('w:hyperlink')
    # Set the relationship ID attribute for the hyperlink
    hyperlink.set(docx.oxml.shared.qn('r:id'), r_id)
    # Create a new run for the drug name text
    new_run = docx.text.run.Run(docx.oxml.shared.OxmlElement('w:r'), paragraph)
    new_run.text = drug_name
    # Apply the hyperlink style to the run
    new_run.style = get_or_create_hyperlink_style(part.document)
    # Append the new run to the hyperlink element
    hyperlink.append(new_run._element)
    # Append the hyperlink element to the paragraph
    paragraph._p.append(hyperlink)
    return hyperlink

# Function to get or create a hyperlink style in the document
def get_or_create_hyperlink_style(document):
    # Check if the "Hyperlink" style exists
    if "Hyperlink" not in document.styles:
        # Create a default character font style if it doesn't exist
        if "Default Character Font" not in document.styles:
            default_character_font = document.styles.add_style("Default Character Font", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
            default_character_font.element.set(docx.oxml.shared.qn('w:default'), "1")
            default_character_font.priority = 1
            default_character_font.hidden = True
            default_character_font.unhide_when_used = True
            del default_character_font
        # Create the "Hyperlink" style
        hyperlink_style = document.styles.add_style("Hyperlink", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
        hyperlink_style.base_style = document.styles["Default Character Font"]
        hyperlink_style.unhide_when_used = True
        hyperlink_style.font.color.rgb = docx.shared.RGBColor(0x05, 0x63, 0xC1)
        hyperlink_style.font.underline = True
        del hyperlink_style
    return "Hyperlink"

# Function to extract drug names from a string description using regex
def extract_hyperlink_data(drug_des):
    pattern = re.compile(r'Drug: (\w+)', re.IGNORECASE)
    matches = re.findall(pattern, drug_des)
    return matches

# Function to set the color of a run in the document
def set_run_color(run, color):
    font = run.font
    font.color.rgb = color

# Function to generate possible fusion gene combinations
def generate_fusion_combinations(gene_names):
    fusion_combinations = set()
    fusion_combinations.add(gene_names[0] + ' Fusion')
    fusion_combinations.add(gene_names[1] + ' Fusion')
    fusion_combinations.add('-'.join(gene_names) + ' Fusion')
    fusion_combinations.add('-'.join(reversed(gene_names)) + ' Fusion')
    return fusion_combinations

# Function to add gene summary text and other relevant information to the document
def add_gene_summary_text(document, gene_heading, ncbi_summary, gene_roles, gene_ckb_des, my_cancer_des, altered_var, oncokb, prognosis_pubmed_des, prognosis_germline_des, approved_therapy_des, drug_des):
    global dna_sequence_variants_merged_with_filter_df, copy_number_variations_merged_with_filter_df, fusion_merged_with_filter_df
    #Gene Heading
    p = document.add_paragraph()
    p.add_run(gene_heading).bold = True
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    for run in p.runs:
        run.font.name = 'Times New Roman'
        run.font.size = Pt(12) 
    #Gene Summary
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Gene Summary").bold = True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(ncbi_summary):
        ncbi_summary = 'NCBI summary information could not be found in the in-house database'
    ncbi_summary = re.sub(r'\[provided by RefSeq, [A-Za-z]+ \d+\]', '', ncbi_summary)
    p.add_run(ncbi_summary)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
        
    #Altered variant
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Altered variant").bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run(altered_var)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'

    p = document.add_paragraph('Link(s) to OncoKB for the variant:\n')
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    hyperlink_text = oncokb.replace('-', '/').replace(' ', '/')
    hyperlink_url = f'https://www.oncokb.org/gene/{hyperlink_text}'
    add_hyperlink(p, oncokb, hyperlink_url)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'

    if 'Fusion' in gene_heading:
        gene_names = gene_heading.replace(' Fusion', '').split('-')
        fusion_combinations = generate_fusion_combinations(gene_names)
        fusion_combinations.discard(' '.join(gene_names) + ' Fusion')
        fusion_combinations.discard(' '.join(reversed(gene_names)) + ' Fusion')
        # Sort fusion combinations by length and then alphabetically
        fusion_combinations = sorted(fusion_combinations, key=lambda x: (len(x.split()[0]), x))
        
        for fusion_heading in fusion_combinations:
            hyperlink_text = fusion_heading.replace('-', '/').replace(' ', '/')
            hyperlink_url = f'https://www.oncokb.org/gene/{hyperlink_text}'
            add_hyperlink(p, fusion_heading, hyperlink_url)
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p = doc.add_paragraph('\n')
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
    
    # Gene disease association
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Gene -Disease association').bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run(gene_roles)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(gene_ckb_des):
        gene_ckb_des = 'CKB-core information could not be found in the in-house database'
    p.add_run(gene_ckb_des)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(my_cancer_des):
        my_cancer_des = 'There is no My cancer genome information for this gene'
    else:
        my_cancer_des = re.sub(r'\[(\d+)\]', r'[My cancer genome]', my_cancer_des)
        my_cancer_des = str(my_cancer_des)
    p.add_run(my_cancer_des)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'

    #Prognostic significance
    if prognosis_pubmed_des is not None:
        prognosis_pubmed_des = str(prognosis_pubmed_des)
        pattern = re.compile(r'\(PMID: \d+\)\.')
        prognosis_pubmed_des = re.sub(pattern, lambda x: x.group() + '\n', prognosis_pubmed_des)
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('Prognostic significance').bold=True
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name ='Times New Roman'
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        if prognosis_pubmed_des.lower() == 'nan':
            prognosis_pubmed_des = 'Prognostic significance information could not be found in the in-house database'
        p.add_run(prognosis_pubmed_des)
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
        if prognosis_germline_des != 'nan':
            #print(prognosis_germline_des)
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run('Prognostic significance of germline mutation').bold=True
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name ='Times New Roman'

            if prognosis_germline_des == 'nan':
                prognosis_germline_des = 'ACMG recommendations are not relevant for this case'
            
            lines = prognosis_germline_des.split('\n')
            for line in lines:
                if line.startswith("*Note: Germline testing is recommended to determine if the identified pathogenic"):
                    p = document.add_paragraph()
                    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
                    p.paragraph_format.left_indent = Cm(2.5)
                    p.paragraph_format.right_indent = Cm(2.5)
                    p.add_run(line).bold = True
                    for run in p.runs:
                        run.font.size = Pt(11)
                        run.font.name = 'Times New Roman'
                else:
                    p = document.add_paragraph()
                    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
                    p.paragraph_format.left_indent = Cm(2.5)
                    p.paragraph_format.right_indent = Cm(2.5)
                    p.add_run(line)
                    for run in p.runs:
                        run.font.size = Pt(11)
                        run.font.name = 'Times New Roman'
                
    #Therapy
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Therapy' + '\n' + 'Approved therapy').bold = True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    if approved_therapy_des == 'nan':
        approved_therapy_des = 'Biomarker directed therapies could not be found in MyCancerGenome'
        p.add_run(approved_therapy_des)
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
    else:
        lines = approved_therapy_des.split('\n')
        color_mapping = {
        'MediumPurple': RGBColor(147, 112, 219),
        'Pink': RGBColor(255, 20, 147),
        'Green': RGBColor(0, 128, 0),
        'Red': RGBColor(255, 0, 0),
        'Black': RGBColor(0, 0, 0)}
        current_color = None
        for line in lines:
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(line)
            if line.startswith('Biomarker Criteria:'):
                current_color = color_mapping['Green']
            elif line.startswith('Predicted Response:'):
                current_color = color_mapping['Red']
            elif line.startswith('Clinical Setting(s):'):
                current_color = color_mapping['Black']
            elif line.startswith('Note:'):
                current_color = color_mapping['Black']
            elif line.startswith('Drug(s)'):
                current_color = color_mapping['MediumPurple']
            elif line.startswith('Cancer:'):
                current_color = color_mapping['Pink']
            for run in p.runs:
                run.font.size = Pt(11)
                run.font.name = 'Times New Roman'
                set_run_color(run, current_color)

    #In trial therapy
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.paragraph_format.space_after = Pt(0) 
    p.add_run('In trial therapy').bold = True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Therapies are being investigated in the clinical trials for patients of xxx or solid tumors with XXX mutation. The patient may be eligible to enroll in these clinical trials. Please see the “Clinical trial” section for more details.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
        
    #Drug resistance information
    drug_des=str(drug_des)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Drug resistance information').bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    if drug_des == 'nan':
        drug_des = 'Drug resistance information could not be found in the in-house database'
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(drug_des)
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name ='Times New Roman'
    else:
        p.add_run('\n')
        for line in drug_des.split('\n'):
            if line.startswith('Drug:'):
                drug_name = line.split('Drug:')[1].strip()
                drug = drug_name.replace(" ","+")
                hyperlink_run = add_hyperlink(p, drug_name, f"https://www.google.com/search?q={drug}+mechanism+of+action")
                p.add_run(line.split(drug_name)[1])
            else:
                p = document.add_paragraph()
                p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
                p.paragraph_format.left_indent = Cm(2.5)
                p.paragraph_format.right_indent = Cm(2.5)
                p.add_run(line)
                for run in p.runs:
                    run.font.size = Pt(11)
                    run.font.name ='Times New Roman'
    p = document.add_paragraph()

In [25]:
def gene_variant_description(document):
    print('In Gene and Variant Description')
    '''
    This is for the section Gene description
    This code processes dna sequence, gene fusion and copy number data by accumulating summaries, roles, descriptions, prognostic significances, 
    and drug descriptions for each gene, normalizing and filtering the information, and then compiling detailed descriptions
    '''
    global dna_sequence_variants_merged_with_filter_df, copy_number_variations_merged_with_filter_df, fusion_merged_with_filter_df, patient_info
    ncbi_df = pd.read_excel(DB_PATH+NCBI_INFO_DB_FILE)
    ncbi_df.drop(['GC_Name', 'GC_Aliases', 'Gene_ID', 'NCBI_Official_Symbol', 'NCBI_Full_Name', 'NCBI_Aliases'], axis=1, inplace=True)
    ncbi_df.rename(columns={'Gene_Name': 'Gene'}, inplace=True)
    ckbcore_df = pd.read_excel(DB_PATH+CKB_CORE_DB_FILE)
    ckbcore_df.rename(columns={'Gene name': 'Gene'}, inplace=True)
    drug_res_df = pd.read_excel(DB_PATH+DRUG_RESISTENCE_DB_FILE)
    drug_res_df['G-KnowMe_Main tumor type'] = drug_res_df['G-KnowMe_Main tumor type'].str.lower()
    tumor_types = [tumor.lower() for tumor in patient_info.get('Main Tumor Type', '').split(', ')]
    prognosis_df = pd.read_excel(DB_PATH+PROGNOSIS_DB_FILE)
    prognosis_df['G-KnowMe_Main tumor type']=prognosis_df['G-KnowMe_Main tumor type'].str.lower()

    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(9.81)
    new_height = Cm(0.56)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Gene_variant_des.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    
    prognosis_des='nan'
    approved_therapy_des = 'nan'
    prognosis_germline_des = 'nan'
    drug_des = 'nan'
    
    #DNA SEQUENCE DATA DESCRIPTION
    try:
        if not dna_sequence_variants_merged_with_filter_df.empty:
            #print('-------dna_sequence_variants_merged_with_filter_df------')
            dna_sequence_variants_df = dna_sequence_variants_merged_with_filter_df.copy()
            dna_sequence_variants_df['Gene_Heading'] = dna_sequence_variants_df['Gene'] + '\n' + dna_sequence_variants_df['Genomic alterations']
            
            #Merge with in-house ncbi data on column Gene
            dna_sequence_variants_df = dna_sequence_variants_df.merge(ncbi_df, on='Gene', how='left', suffixes=('', '_y'))
            dna_sequence_variants_df.drop(dna_sequence_variants_df.filter(regex='_y$').columns, axis=1, inplace=True)
            #print('after ncbi')
            #print(dna_sequence_variants_df)
            
            #Merge with in-house ckbcore data on column Gene
            dna_sequence_variants_df = dna_sequence_variants_df.merge(ckbcore_df, on='Gene', how='left', suffixes=('', '_y'))
            dna_sequence_variants_df.drop(dna_sequence_variants_df.filter(regex='_y$').columns, axis=1, inplace=True)
            #print('after ckb')
            #print(dna_sequence_variants_df)
            
            #Merge with in-house prognosis data on column Gene
            prognosis_filtered_tumor_types = prognosis_df[(prognosis_df['Gene'].isin(dna_sequence_variants_df['Gene']))&(prognosis_df['G-KnowMe_Main tumor type'].isin(tumor_types))]
            prognosis_filtered_solid_tumors = prognosis_df[(prognosis_df['Gene'].isin(dna_sequence_variants_df['Gene']))&(prognosis_df['G-KnowMe_Main tumor type'] == 'Solid tumours/Pan-Cancer')]
            prognosis_filtered_df = pd.concat([prognosis_filtered_tumor_types, prognosis_filtered_solid_tumors])
            prognosis_filtered_df =prognosis_filtered_df.groupby('Gene')['Prognostic significance'].apply(lambda x: '\n'.join(x)).reset_index()
            dna_sequence_variants_df = dna_sequence_variants_df.merge(prognosis_filtered_df, on='Gene', how='left', suffixes=('', '_y'))
            dna_sequence_variants_df.drop(dna_sequence_variants_df.filter(regex='_y$').columns, axis=1, inplace=True)
            #print('after prognosis')
            #print(dna_sequence_variants_df)
            
            #Merge with in-house drug resistance data on column Gene
            drug_res_filtered_tumor_types = drug_res_df[(drug_res_df['Gene'].isin(dna_sequence_variants_df['Gene']))&(drug_res_df['G-KnowMe_Main tumor type'].isin(tumor_types))]
            drug_res_filtered_solid_tumors = drug_res_df[(drug_res_df['Gene'].isin(dna_sequence_variants_df['Gene']))&(drug_res_df['G-KnowMe_Main tumor type'] == 'solid tumours')]
            drug_res_filtered_df = pd.concat([drug_res_filtered_tumor_types, drug_res_filtered_solid_tumors])
            drug_res_filtered_df["Drug_name_description"] = 'Drug: '+ drug_res_filtered_df["Potential drug \nresistance"].fillna('') + '\n' + drug_res_filtered_df["Description"].fillna('')+ '\n' 
            drug_res_filtered_df = drug_res_filtered_df.groupby('Gene')['Drug_name_description'].apply(lambda x: '\n'.join(x)).reset_index()
            dna_sequence_variants_df = dna_sequence_variants_df.merge(drug_res_filtered_df, on='Gene', how='left', suffixes=('', '_y'))
            dna_sequence_variants_df.drop(dna_sequence_variants_df.filter(regex='_y$').columns, axis=1, inplace=True)
            #print('after drug')
            #print(dna_sequence_variants_df)
            
            for col in dna_sequence_variants_df.columns:
                if dna_sequence_variants_df[col].apply(lambda x: isinstance(x, list)).any():
                    #Convert all lists in the column to tuples to make them hashable for drop_duplicates
                    dna_sequence_variants_df[col] = dna_sequence_variants_df[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)
            dna_sequence_variants_df.drop_duplicates(inplace=True)
            
            for index, row in dna_sequence_variants_df.iterrows():
                #print('inside loop')
                 # Get the gene name from the 'Gene_Heading' column
                dna_gene_heading = row['Gene_Heading']
                gene =row['Gene']
                
                # Get the gene summary from the 'NCBI_Summary' column
                ncbi_summary = row['NCBI_Summary']
                
                # Get the gene roles from the 'Gene Role' column (handle missing values)
                gene_roles = row.get('Gene Role', None)
                if gene_roles is not None and pd.notna(gene_roles):
                    # Extract and clean gene roles information
                    gene_roles = gene_roles.split(' (')[0].replace('Both: ', '')
                    gene_roles = re.sub(r'\([^)]*\)', '', gene_roles)
                else:
                    # Provide default text if gene roles information is missing
                    gene_roles = 'There is no Gene role information for this gene'
                gene_ckb_des = row['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]']
                df=pd.DataFrame()
                
                # Check if the gene name contains spaces and adjust the format accordingly
                if ' ' in gene.strip():
                    gene=gene.lower()
                    gene=gene.replace(' ', '-')
                    df=pd.concat([df,For_Alterations(gene)])
                elif ' ' not in gene.lower().strip():
                    df=pd.concat([df,For_Genes(gene)])
                    
                #Retrieve gene role information from the DataFrame or provide default text
                my_cancer_des = df['Gene_Role'].iloc[0].replace('\n', '').replace('  ', '') if not df.empty else 'There is no My cancer genome information for this gene'
                # Create a condition to filter rows with non-empty Drugs, Cancer, and Biomarker_Criteria columns
                condition = (~df['Drugs'].eq('')) & (~df['Cancer'].eq('')) & (~df['Biomarker_Criteria'].eq(''))
                df['my_cancer_data'] = ''
                df.loc[condition, 'my_cancer_data'] = 'Drug(s): ' + df.loc[condition, 'Drugs'] + '\n' + 'Cancer: ' + df.loc[condition, 'Cancer'] + '\n' + df.loc[condition, 'Biomarker_Criteria']
                df = df[df['Gene_Name'] == gene]
                # Compile approved therapy descriptions or provide default text if not available
                approved_therapy_des = '\n'.join(df['my_cancer_data']) if not df.empty and 'my_cancer_data' in df else 'Biomarker directed therapies could not be found in MyCancerGenome'
                if pd.isna(approved_therapy_des) or approved_therapy_des == '':
                    approved_therapy_des = 'Biomarker directed therapies could not be found in MyCancerGenome' 
                
                #Prognostic significance from prognosis in-house database
                prognosis_des=row['Prognostic significance']
                
                #Para for altered variant section
                altered_var = row['altered_variant_para']
                #Used to create link for OncoKB
                oncokb = row['OncoKB_input']
                
                #Basic data extraction for checking if the varaint is valid for ACMG clinvar statement
                variant_classification = row['Classification']
                vaf_percentage = row['Allele Frequency']
                vaf_percentage = vaf_percentage.replace('%', '').strip()
                if ('Pathogenic' in variant_classification or 'Likely Pathogenic' in variant_classification) and float(vaf_percentage) >= 20:
                    # ACMG data is extracted if the variant_classification is pathogenic or likely pathogenic variants with a VAF of at least 20%
                    medgen_df = ACMG_dataextraction(gene)
                    if medgen_df is not None:
                        if not medgen_df.empty:
                            matching_rows = medgen_df[medgen_df['Gene_Name'] == gene]
                            if not matching_rows.empty and 'disease_characteristics' in matching_rows:
                                prognosis_germline_des = ' '.join(matching_rows['disease_characteristics'])
                                prognosis_germline_des += f"\n*Note: Germline testing is recommended to determine if the identified pathogenic {gene} mutation is germline or somatic."
                        else:
                            prognosis_germline_des = 'ACMG recommendations are not relevant for this case' 
                    else:
                        prognosis_germline_des = 'ACMG recommendations are not relevant for this case' 
                elif ('uncertain significance' in variant_classification.lower() or not re.search(r'[a-zA-Z]', variant_classification)) and float(vaf_percentage) >= 20:
                    # ACMG data is extracted if the variant_classification is uncertain significance (VUS) with a VAF of at least 20%
                    medgen_df = ACMG_dataextraction(gene)
                    if medgen_df is not None:
                        if not medgen_df.empty:
                            matching_rows = medgen_df[medgen_df['Gene_Name'] == gene]
                            if not matching_rows.empty and 'disease_characteristics' in matching_rows:
                                prognosis_germline_des = ' '.join(matching_rows['disease_characteristics'])
                                prognosis_germline_des += f"\n*This patient has a VUS-Deleterious mutation in {gene} with {vaf_percentage}% variant allele frequency."
                        else:
                            prognosis_germline_des = 'ACMG recommendations are not relevant for this case' 
                    else:
                        prognosis_germline_des = 'ACMG recommendations are not relevant for this case'
                else:
                    # Default case for all other variant classifications or VAF percentages
                    prognosis_germline_des = 'ACMG recommendations are not relevant for this case'
                    
                #Drug_name_description from drug resistance in-house database
                drug_des=row['Drug_name_description']
                add_gene_summary_text(document, dna_gene_heading, ncbi_summary, gene_roles, gene_ckb_des, my_cancer_des, altered_var, oncokb, prognosis_des, prognosis_germline_des, approved_therapy_des, drug_des)
    except:
         print("DataFrame 'dna_sequence_variants_merged_with_filter_df' is not available.")    
    print('Done with Dna sequence varaints')
    
    try: 
        if not copy_number_variations_merged_with_filter_df is None or copy_number_variations_merged_with_filter_df.empty:
            copy_number_variants_df=copy_number_variations_merged_with_filter_df.copy()
            copy_number_variants_df['Gene_Heading'] = copy_number_variants_df['Gene'] + '\n' + copy_number_variants_df['CytoBand'] + '\n' + copy_number_variants_df['Oncomine Variant Class']
            copy_number_variants_df['Genomic Alteration']=copy_number_variants_df['Gene']+' '+copy_number_variants_df['Oncomine Variant Class']
            
            #Merge with in-house ncbi data on column Gene
            copy_number_variants_df = copy_number_variants_df.merge(ncbi_df, on='Gene', how='left', suffixes=('', '_y'))
            copy_number_variants_df.drop(copy_number_variants_df.filter(regex='_y$').columns, axis=1, inplace=True)
            
            #Merge with in-house ckbcore data on column Gene
            copy_number_variants_df = copy_number_variants_df.merge(ckbcore_df, on='Gene', how='left', suffixes=('', '_y'))
            copy_number_variants_df.drop(copy_number_variants_df.filter(regex='_y$').columns, axis=1, inplace=True)
            
            #Merge with in-house prognosis data on column Gene
            prognosis_filtered_tumor_types = prognosis_df[(prognosis_df['Gene'].isin(copy_number_variants_df['Gene']))&(prognosis_df['G-KnowMe_Main tumor type'].isin(tumor_types))]
            prognosis_filtered_solid_tumors = prognosis_df[(prognosis_df['Gene'].isin(copy_number_variants_df['Gene']))&(prognosis_df['G-KnowMe_Main tumor type'] == 'solid tumours')]
            prognosis_filtered_df = pd.concat([prognosis_filtered_tumor_types, prognosis_filtered_solid_tumors])
            prognosis_filtered_df =prognosis_filtered_df.groupby('Gene')['Prognostic significance'].apply(lambda x: '\n'.join(x)).reset_index()
            copy_number_variants_df = copy_number_variants_df.merge(prognosis_filtered_df, on='Gene', how='left', suffixes=('', '_y'))
            copy_number_variants_df.drop(copy_number_variants_df.filter(regex='_y$').columns, axis=1, inplace=True)
            
            #Merge with in-house drug resistance data on column Gene
            drug_res_filtered_tumor_types = drug_res_df[(drug_res_df['Gene'].isin(copy_number_variants_df['Gene']))&(drug_res_df['G-KnowMe_Main tumor type'].isin(tumor_types))]
            drug_res_filtered_solid_tumors = drug_res_df[(drug_res_df['Gene'].isin(copy_number_variants_df['Gene']))&(drug_res_df[''] == 'solid tumours')]
            drug_res_filtered_df = pd.concat([drug_res_filtered_tumor_types, drug_res_filtered_solid_tumors])
            drug_res_filtered_df["Drug_name_description"] = 'Drug: '+ drug_res_filtered_df["Potential drug \nresistance"].fillna('') + '\n' + drug_res_filtered_df["Description"].fillna('')+ '\n' 
            drug_res_filtered_df = drug_res_filtered_df.groupby('Gene')['Drug_name_description'].apply(lambda x: '\n'.join(x)).reset_index()
            copy_number_variants_df = copy_number_variants_df.merge(drug_res_filtered_df, on='Gene', how='left', suffixes=('', '_y'))
            copy_number_variants_df.drop(copy_number_variants_df.filter(regex='_y$').columns, axis=1, inplace=True)
          
            for index, row in copy_number_variants_df.iterrows():
                # Get the gene name from the 'Gene_Heading' column
                copy_gene_heading = row['Gene_Heading']
                
                # Get the variant class and convert it to lowercase
                variant_class = row['Oncomine Variant Class'].lower()
                # Create a varaible gene based on the variant class 
                if variant_class == 'deletion':
                    gene= row['Gene'] + '-Loss' # Add "-Loss" for deletions
                else:
                    gene= row['Gene'] + '-'+row['Oncomine Variant Class'] # Combine gene and variant class
                # Get the gene summary from the 'NCBI_Summary' column
                ncbi_summary = row['NCBI_Summary']
                
                # Get the gene roles from the 'Gene Role' column (handle missing values)
                gene_roles = row.get('Gene Role', None)
                if gene_roles is not None and pd.notna(gene_roles):
                    # Clean up the gene roles text
                    gene_roles = gene_roles.split(' (')[0].replace('Both: ', '')
                    gene_roles = re.sub(r'\([^)]*\)', '', gene_roles)
                else:
                    gene_roles = 'There is no Gene role information for this gene'
                
                # Get the gene description from the 'Gene Description                                                                                            [Source : CKB core https://ckb.jax.org/gene/grid]' column (notice extra spaces)
                gene_ckb_des = row['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]']
                df=pd.DataFrame()
                # Convert the gene name to lowercase and replace spaces with hyphens
                gene=gene.lower()
                gene=gene.replace(' ', '-')
                df=pd.concat([df,For_Alterations(gene)])
                # Get the first row's 'Gene_Role' value from the df (assuming For_Alterations populates it), clean it up (handle empty df)
                my_cancer_des = df['Gene_Role'].iloc[0].replace('\n', '').replace('  ', '') if not df.empty else 'There is no My cancer genome information for this gene'
                # Filter the df based on non-empty values in 'Drugs', 'Cancer', and 'Biomarker_Criteria' columns
                condition = (~df['Drugs'].eq('')) & (~df['Cancer'].eq('')) & (~df['Biomarker_Criteria'].eq(''))
                df['my_cancer_data'] = ''
                df.loc[condition, 'my_cancer_data'] = 'Drug(s): ' + df.loc[condition, 'Drugs'] + '\n' + 'Cancer: ' + df.loc[condition, 'Cancer'] + '\n' + df.loc[condition, 'Biomarker_Criteria']
                df = df[df['Gene_Name'] == gene]
                # Compile approved therapy descriptions or provide default text if not available
                approved_therapy_des = '\n'.join(df['my_cancer_data']) if not df.empty and 'my_cancer_data' in df else 'Biomarker directed therapies could not be found in MyCancerGenome'
                if pd.isna(approved_therapy_des) or approved_therapy_des == '':
                    approved_therapy_des = 'Biomarker directed therapies could not be found in MyCancerGenome'
                
                # Para for altered variant section
                altered_var = row['altered_variant_para'] 
                # Used to create link for OncoKB
                oncokb = row['Genomic Alteration']
                # Prognostic significance from prognosis in-house database
                prognosis_des=row['Prognostic significance']
                # Drug_name_description from drug resistance in-house database
                drug_des=row['Drug_name_description']
                add_gene_summary_text(document, copy_gene_heading, ncbi_summary, gene_roles, gene_ckb_des, my_cancer_des, altered_var, oncokb, prognosis_des, 'ACMG recommendations are not relevant for this case', approved_therapy_des, drug_des)
    
    except:
         print("DataFrame 'copy_number_variations_merged_with_filter_df' is not available.") 
        
    print('Done with copy number variants')
    try:
        if not fusion_merged_with_filter_df.empty:
            gene_fusions_df=fusion_merged_with_filter_df.copy()
            gene_fusions_df['Gene_Heading'] = gene_fusions_df['Genes (Exons)'] + ' Fusion'
            gene_fusions_df['Genes'] = gene_fusions_df['Genes'].astype(str)
            gene_fusions_df['Gene'] = gene_fusions_df['Genes'].apply(lambda x: x.replace('::', ', '))
            gene_fusions_df = gene_fusions_df.assign(Gene=gene_fusions_df['Gene'].str.split(', ')).explode('Gene')
            
            #Merge with in-house ncbi data on column Gene
            gene_fusions_df = gene_fusions_df.merge(ncbi_df, on='Gene', how='inner', suffixes=('', '_y'))
            gene_fusions_df.drop(gene_fusions_df.filter(regex='_y$').columns, axis=1, inplace=True)

            #Merge with in-house ckbcore data on column Gene
            gene_fusions_df = gene_fusions_df.merge(ckbcore_df, on='Gene', how='left', suffixes=('', '_y'))
            gene_fusions_df.drop(gene_fusions_df.filter(regex='_y$').columns, axis=1, inplace=True)
            
            #Merge with in-house prognosis data on column Gene
            prognosis_filtered_tumor_types = prognosis_df[(prognosis_df['Gene'].isin(gene_fusions_df['Gene']))&(prognosis_df['G-KnowMe_Main tumor type'].isin(tumor_types))]
            prognosis_filtered_solid_tumors = prognosis_df[(prognosis_df['Gene'].isin(gene_fusions_df['Gene']))&(prognosis_df['G-KnowMe_Main tumor type'] == 'solid tumours')]
            prognosis_filtered_df = pd.concat([prognosis_filtered_tumor_types, prognosis_filtered_solid_tumors])
            prognosis_filtered_df =prognosis_filtered_df.groupby('Gene')['Prognostic significance'].apply(lambda x: '\n'.join(x)).reset_index()
            gene_fusions_df = gene_fusions_df.merge(prognosis_filtered_df, on='Gene', how='left', suffixes=('', '_y'))
            gene_fusions_df.drop(gene_fusions_df.filter(regex='_y$').columns, axis=1, inplace=True)
            
            #Merge with in-house drug resistance data on column Gene
            drug_res_filtered_tumor_types = drug_res_df[(drug_res_df['Gene'].isin(gene_fusions_df['Gene']))&(drug_res_df['G-KnowMe_Main tumor type'].isin(tumor_types))]
            drug_res_filtered_solid_tumors = drug_res_df[(drug_res_df['Gene'].isin(gene_fusions_df['Gene']))&(drug_res_df['G-KnowMe_Main tumor type'] == 'solid tumours')]
            drug_res_filtered_df = pd.concat([drug_res_filtered_tumor_types, drug_res_filtered_solid_tumors])
            drug_res_filtered_df["Drug_name_description"] = 'Drug: '+ drug_res_filtered_df["Potential drug \nresistance"].fillna('') + '\n' + drug_res_filtered_df["Description"].fillna('')+ '\n' 
            drug_res_filtered_df = drug_res_filtered_df.groupby('Gene')['Drug_name_description'].apply(lambda x: '\n'.join(x)).reset_index()
            gene_fusions_df = gene_fusions_df.merge(drug_res_filtered_df, on='Gene', how='left', suffixes=('', '_y'))
            gene_fusions_df.drop(gene_fusions_df.filter(regex='_y$').columns, axis=1, inplace=True)
            
            fusion_df = gene_fusions_df[['Gene', 'NCBI_Summary', 'Gene Role', 'Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]', 'Prognostic significance', 'Drug_name_description']].copy()
            fusion_df.drop_duplicates(subset=['Gene'], keep='first', inplace=True)
            gene_fusions_df.drop(['Gene', 'NCBI_Summary', 'Gene Role', 'Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]', 'Prognostic significance', 'Drug_name_description'], axis=1, inplace=True)
            gene_fusions_df.drop_duplicates(subset=['Gene_Heading'], keep='first', inplace=True)
            for index, row in gene_fusions_df.iterrows():
                fusion_heading = row['Gene_Heading']
                altered_var = row['altered_variant_para']
                gene = fusion_heading 
                df=pd.DataFrame()
                gene=gene.lower()
                gene=gene.replace(' ', '-')
                df=pd.concat([df,For_Alterations(gene)])
                my_cancer_des = df['Gene_Role'].iloc[0].replace('\n', '').replace('  ', '') if not df.empty else 'There is no My cancer genome information for this gene'
                condition = (~df['Drugs'].eq('')) & (~df['Cancer'].eq('')) & (~df['Biomarker_Criteria'].eq(''))
                df['my_cancer_data'] = ''
                df.loc[condition, 'my_cancer_data'] = 'Drug(s): ' + df.loc[condition, 'Drugs'] + '\n' + 'Cancer: ' + df.loc[condition, 'Cancer'] + '\n' + df.loc[condition, 'Biomarker_Criteria']
                df = df[df['Gene_Name'] == gene]
                approved_therapy_des = '\n'.join(df['my_cancer_data']) if not df.empty and 'my_cancer_data' in df else 'Biomarker directed therapies could not be found in MyCancerGenome'
                if pd.isna(approved_therapy_des) or approved_therapy_des == '':
                    approved_therapy_des = 'Biomarker directed therapies could not be found in MyCancerGenome'
                ncbi_summary_accumulator = ''
                gene_roles_accumulator = ''
                gene_ckb_des_accumulator = ''
                prognosis_des_accumulator = ''
                drug_des_accumulator = ''
                
                for index, row in fusion_df.iterrows():
                    gene = row['Gene']
                    if not pd.isna(row['NCBI_Summary']):
                        ncbi_summary_accumulator += gene + '\n' + row['NCBI_Summary'] + '\n'
                    else:
                        ncbi_summary_accumulator += gene + '\n' + 'NCBI Summary information could not be found in the in-house database' + '\n'

                    if not pd.isna(row['Gene Role']):
                        gene_roles = row.get('Gene Role', None)
                        if gene_roles is not None and pd.notna(gene_roles):
                            gene_roles = gene_roles.split(' (')[0].replace('Both: ', '')
                            gene_roles = re.sub(r'\([^)]*\)', '', gene_roles)
                            gene_roles_accumulator += gene_roles + '\n'
                    else:
                        gene_roles_accumulator += 'There is no Gene role information for this gene' + '\n'

                    gene_ckb_des_accumulator += gene + '\n' + row['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]'] + '\n'

                    if not pd.isna(row['Prognostic significance']):
                        prognosis_des_accumulator += gene + '\n' + row['Prognostic significance'] + '\n'
                    else:
                        prognosis_des_accumulator += gene + '\n' + 'Prognostic significance information could not be found in the in-house database' + '\n'

                    if not pd.isna(row['Drug_name_description']):
                        drug_des_accumulator += gene + '\n' + row['Drug_name_description'] + '\n'
                    else:
                        drug_des_accumulator += gene + '\n' + 'Drug resistance information could not be found in the in-house database' + '\n'

                add_gene_summary_text(document, fusion_heading, ncbi_summary_accumulator, gene_roles_accumulator, gene_ckb_des_accumulator, my_cancer_des, altered_var, '', prognosis_des_accumulator, 'ACMG recommendations are not relevant for this case', approved_therapy_des, drug_des_accumulator)

    except:
         print("DataFrame 'fusion_merged_with_filter_df' is not available.") 
    print('Done with fusion variants')

In [26]:
def hrr_table(document, data):
    print('Adding HRR table')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(1.4)
        section.right_margin = Cm(2)

    table = document.add_table(rows=len(data) + 1, cols=2)  # Add 1 for header row
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(8.25)
    table.columns[1].width = Cm(8.25)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style = 'Table Grid'
    table.style.font.name = 'Times New Roman'
    table.style.font.size = Pt(11)

    header_row = table.rows[0].cells
    header_row[0].text = "Gene/Genomic Alteration"
    header_row[1].text = "Findings"
    
    for i, (index, row_data) in enumerate(data.iterrows()):
        row_cells = table.rows[i + 1].cells  # Add 1 to skip header row
        row_cells[0].text = row_data["Gene/Genomic Alteration"]
        row_cells[1].text = row_data["Finding"]
        
    for cell in table.rows[0].cells:
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.bold = True
        cell.paragraphs[0].runs[0].font.size = Pt(11)
        
    return table

In [27]:
def hrr_genes(document):
    global hrr_df
    print('In HRR Section')
    # Set margins for all sections of the document
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    
    # Define the dimensions of the image
    new_width = Cm(12.17)
    new_height = Cm(0.85)
    
    # Add a paragraph for the image and set the indentation
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    
    # Define the path to the image and add the image to the paragraph
    image_path = os.path.join("Heading_Images", 'HRR_genes.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    
    # Extract the LOH score if available
    loh_score = hrr_df.loc[hrr_df['Gene/Genomic Alteration'] == 'LOH percentage', 'Finding'].str.extract(r'(\d+\.\d+)').iloc[0, 0] if not hrr_df.loc[hrr_df['Gene/Genomic Alteration'] == 'LOH percentage', 'Finding'].empty else None
    loh_score_str = str(loh_score) + '%' if loh_score is not None else None
    
    # Add a paragraph with introductory text about HRR genes
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Homologous recombination repair (HRR) pathway genes play a vital role in maintaining genome stability and tumor suppression. Alterations in HRR genes can lead to genome instability which in turn may increase the risk of developing tumors. Thus knowing the alterations in HRR genes can act as potential biomarkers to decide the personalized therapy to be undertaken.')
    
    # Set the font style for the paragraph
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    
    # Initialize the note_text variable
    note_text = ''
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    
    # If LOH score is available, create the note text
    if loh_score_str:
        note_text = f'Note: This patient shows {loh_score_str} of LOH within the HRR gene panel that includes BRCA1/2. To further confirm the HRR/HRD status, assessment of HRR/HRD using orthogonal methods is recommended.'
    
    # Add the note text to the paragraph and set the font style
    for char in note_text:
        run = p.add_run(char)
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
        if char in loh_score_str:
            run.bold = True
    
    # If LOH score is present, add additional information and notes
    if loh_score_str:
        hrr_table(document, hrr_df)  # Call the hrr_table function to add HRR table
        
        # Add note about the total number of HRR genes
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('*Note: From the total of 46 HRR genes (including BRCA1 and BRCA2) covered in Oncomine Comprehensive Assay Plus, these alterations are identified. Confirmation with other orthogonal methods is recommended.')
        
        # Set the font style for the note
        for run in p.runs:
            run.font.size = Pt(10)
            run.font.name = 'Times New Roman'
        
        # Add note about the definition of HRR genes
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('*Note: Homologous recombination repair (HRR) genes were defined from published evidence in relevant therapies, clinical guidelines, as well as clinical trials, and include - BRCA1, BRCA2, ATM, BARD1, BLM, BRIP1, CDK12, CHEK1, CHEK2, FANCL, NBN, PALB2, POLD1, POLE, PPP2R2A, RAD51B, RAD51C, RAD51D, and RAD54L.')
        
        # Set the font style for the note
        for run in p.runs:
            run.font.size = Pt(10)
            run.font.name = 'Times New Roman'

        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('Therapy (This patient is not a confirmed case of HRD)')
            
        # Set the font style for the note
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name = 'Times New Roman'

        if '0' in loh_score_str:
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run('Note: For this patient, LOH was not detected')
        
            # Set the font style for the note
            for run in p.runs:
                run.font.size = Pt(10)
                run.font.name = 'Times New Roman'
            
                
    
    # If no LOH score, add a different note indicating no HRR genes
    else:
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('Note: For this patient, LOH was not detected')
        
        # Set the font style for the note
        for run in p.runs:
            run.font.size = Pt(10)
            run.font.name = 'Times New Roman'


In [28]:
#Creates an empty clinical trial table 
def clinical_trial_table(document):
    print('Adding Clinical trial table')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
        
    line1_paragraph = document.add_paragraph("Following ongoing clinical trials are relevant for the patient's diagnosis and genomic profile")
    line1_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line1_paragraph.paragraph_format.left_indent = Cm(2.5)
    line1_paragraph.paragraph_format.right_indent = Cm(2.5)
    line1_paragraph.paragraph_format.space_after = Cm(0)
    for run in line1_paragraph.runs:
        run.font.size = Pt(13)
        run.font.name = 'Times New Roman'

    line2_paragraph = document.add_paragraph("(Additional trials that are investigating immunotherapies relevant for the patient’s diagnosis or relevant trials that are recruiting patients in India are also listed below)")
    line2_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line2_paragraph.paragraph_format.left_indent = Cm(2.5)
    line2_paragraph.paragraph_format.right_indent = Cm(2.5)
    line2_paragraph.paragraph_format.space_after = Cm(0)
    for run in line2_paragraph.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    
    p = document.add_paragraph()
    table = document.add_table(rows=16, cols=6)  
    table.style = 'Table Grid'
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.autofit = False
    table.allow_autofit = False
    widths = [3.5, 5, 3, 1.85, 3, 2.90]  
    for i, width in enumerate(widths):
        table.cell(0, i).width = Cm(width)
        
    header = table.rows[0].cells
    header[0].text = "Relevant Variant/ Parameter"
    header[1].text = "Clinical trial"
    header[2].text = "Intervention"
    header[3].text = "Phase"
    header[4].text = "Trial identifier"
    header[5].text = "Recruiting location"

    for cell in header:
        cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].bold = True

    for row in table.rows[1:]:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(10)
                    run.font.name = 'Times New Roman'

    return table


In [29]:
#Empty format is created for the clinical trial section
def clinical_trial(document):
    print('In Clinical Trial section')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(6.46)
    new_height = Cm(0.97)
    paragraph = document.add_paragraph()
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Clinical_trials.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    paragraph = document.add_paragraph()
    clinical_trial_table(document)
    p = document.add_paragraph()
    document.paragraphs[-1].paragraph_format.space_before = Pt(0)
    note1 = document.add_paragraph("Note 1: Confirmation of somatic/germline status of certain variants will be necessary to determine the patient's eligibility for the listed trials.")
    note1.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note1.paragraph_format.left_indent = Cm(2.5)
    note1.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    note1.paragraph_format.line_spacing = 1
    for run in note1.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    note2 = document.add_paragraph("Note 2: Patient’s known treatment history, as per the discharge summary is also taken into consideration while matching these trials.")
    note2.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note2.paragraph_format.left_indent = Cm(2.5)
    note2.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    for run in note2.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    note3 = document.add_paragraph("Note 3:To further determine the patient's eligibility for the trial, careful matching of the inclusion criteria based upon the patient’s condition and other clinical parameters is necessary. Certain comorbidities may change this patient's eligibility for the listed trials.")
    note3.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note3.paragraph_format.left_indent = Cm(2.5)
    note3.paragraph_format.right_indent = Cm(2.5)
    note2.paragraph_format.space_after = Cm(0)
    for run in note3.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'

In [30]:
#Generates quality control table 
def quality_control(document):
    ('Adding Quality control table')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)

    table = document.add_table(rows=2, cols=2)
    table.style = 'Table Grid'
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(4.00)
    table.columns[1].width = Cm(4.00)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style.font.name = 'Times New Roman'
    table.style.font.size = Pt(11)
    table.cell(0, 0).text = 'Mean depth coverage'
    table.cell(0, 1).text = mdc
    table.cell(1, 0).text = 'Target Base coverage at 500X'
    table.cell(1, 1).text = tbc
    return table

In [31]:
#Static Creates the table in the supplementary section
def supplementary_table(document):
    print('Adding supplementary table')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
    table = doc.add_table(rows=11, cols=2)
    table.style = 'Table Grid'
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(3.31)
    table.columns[1].width = Cm(13.36)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.cell(0, 0).merge(table.cell(0, 1)).text = 'Variant Classification'
    table.cell(0, 0).paragraphs[0].runs[0].bold = True
    table.cell(1, 0).merge(table.cell(1, 1)).text = 'Tier I: Variants of Strong clinical significance'
    table.cell(1, 0).paragraphs[0].runs[0].bold = True
    table.cell(2, 0).text = 'Level A'
    table.cell(2, 0).paragraphs[0].runs[0].bold = True
    table.cell(2, 1).text = 'FDA-approved therapy included in professional guidelines'
    table.cell(3, 0).text = 'Level B'
    table.cell(3, 0).paragraphs[0].runs[0].bold = True
    table.cell(3, 1).text = 'Well-powered studies with consensus from experts in the field'
    table.cell(4, 0).merge(table.cell(4, 1)).text = 'Tier II: Variants of Potential Clinical Significance'
    table.cell(4, 0).paragraphs[0].runs[0].bold = True
    table.cell(5, 0).text = 'Level C'
    table.cell(5, 0).paragraphs[0].runs[0].bold = True
    table.cell(5, 1).text = 'FDA-approved therapies for different tumor types or investigational therapies\nMultiple small published studies with some consensus'
    table.cell(6, 0).text = 'Level D'
    table.cell(6, 0).paragraphs[0].runs[0].bold = True
    table.cell(6, 1).text = 'Preclinical trials or a few case reports without consensus'
    table.cell(7, 0).merge(table.cell(7, 1)).text = 'Tier III: Variants of Unknown Clinical Significance'
    table.cell(7, 0).paragraphs[0].runs[0].bold = True
    table.cell(8, 0).merge(table.cell(8, 1)).text = 'Not observed at a significant allele frequency in the general or specific subpopulation databases, or pan-cancer or tumor-specific variant databases. No convincing published evidence of cancer association'
    table.cell(9, 0).merge(table.cell(9, 1)).text = 'Tier IV: Benign or Likely Benign Variants'
    table.cell(9, 0).paragraphs[0].runs[0].bold = True
    table.cell(10, 0).merge(table.cell(10, 1)).text = 'Observed at a significant allele frequency in the general or specific subpopulation databases. No existing published evidence of cancer association'
    for row in table.rows:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                if paragraph.runs:
                    run = paragraph.runs[0]
                    run.font.name = 'Times New Roman'
                    run.font.size = Pt(10)

In [32]:
#Static text Entire supplementary section is created here 
def supplementary_information(document):
    print('In Supplementary Information section')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    p = document.add_paragraph()
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Supplementary Information').bold=True
    for run in p.runs:
        run.font.size = Pt(14)
        run.font.name ='Times New Roman'
        run.font.underline = True
    p = document.add_paragraph()
    new_width = Cm(7.41)
    new_height = Cm(0.65)

    # Test description section
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Test_des.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Comprehensive genomic profiling (CGP) is advancing precision oncology research through the analysis of multiple relevant biomarkers in a single next-generation sequencing (NGS) test. The test can detect all types of single-gene variants, such as single-nucleotide variants (SNVs), insertions and deletions (indels), novel and known fusions,  splice variants, and copy number variants (CNVs), including both copy number gains and losses.A  study potential responses to immunotherapies with measurement of tumor mutational burden (TMB) and predisposition to genetic hypermutability by comparing microsatellite instability (MSI) regions, and analyse mutational signatures for insights into etiological factors in tumorigenesis. It can detect both gene-level and sample-level loss of heterozygosity (LOH) to assess genomic instability and mutations in 46 key genes in the homologous  recombination repair (HRR) pathway.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 

    # Quality Control Statistics section, table gets populated from input oncomine report 
    p = document.add_paragraph()
    new_width = Cm(7.41)
    new_height = Cm(0.71)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Quality_Control_Statics.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    quality_control(document)

    #Genes_analyzed section
    p = document.add_paragraph()
    new_width = Cm(6.49)
    new_height = Cm(0.66)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Genes_analyzed.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed for the Detection of DNA Sequence Variants: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('ABL1, ABL2, ACVR1, ACVR2A, AKT1, AKT2, AKT3, ALK, AMER1, APC, AR, ARAF, ARID1A, ARID1B, ARID2, ASXL1, ASXL2, ATM, ATP1A1, ATR, ATRX, AURKA, AURKC, AXIN1, AXIN2, AXL, B2M, BAP1, BCL2, BCL2L12, BCL6, BCOR, BCR, BLM, BMP5, BRAF, BRCA1, BRCA2, BRIP1, BTK, CACNA1D, CALR, CARD11, CASP8, CBL, CCND1, CCND2, CCND3, CCNE1, CD79B, CDC73, CDH1, CDK4, CDK6, CDKN2A, CDKN2C, CHD4, CHEK2, CIC, CREBBP, CSF1R, CTCF, CTNNB1, CUL1, CUL3, CYP2D6, CYSLTR2, DDR2, DDX3X, DGCR8, DICER1, DNMT3A, DPYD, DROSHA, E2F1, EGFR, EIF1AX, EP300, EPAS1, EPHA2, ERBB2, ERBB3, ERBB4, ERCC2, ERCC5, ERRFI1, ESR1, EZH2, FAM135B, FANCM, FBXW7, FGF7, FGFR1, FGFR2, FGFR3, FGFR4, FLT3, FLT4, FOXA1, FOXL2, FOXO1, FUBP1, GATA2, GATA3, GLI1, GNA11, GNA13, GNAQ, GNAS, GPS2, H2BC5, H3-3A, H3-3B, H3C2, HIF1A, HNF1A, HRAS, ID3, IDH1, IDH2, IKBKB, IL6ST, IL7R, IRF4, IRS4, JAK1, JAK2, JAK3, KDM6A, KDR, KEAP1, KIT, KLF4, KLF5, KMT2B, KMT2D, KNSTRN, KRAS, LARP4B, LATS1, MAGOH, MAP2K1, MAP2K2, MAP2K4, MAP2K7, MAP3K4, MAPK1, MAPK8, MAX, MDM4, MECOM, MED12, MEF2B, MEN1, MET, MGA, MITF, MLH3, MPL, MSH3, MSH6, MTOR, MYC, MYCN, MYD88, MYOD1, NBN, NCOR1, NF1, NF2, NFE2L2, NOTCH1, NOTCH2, NRAS, NSD2, NT5C2, NTRK1, NTRK2, NTRK3, NUP93, PARP1, PAX5, PBRM1, PCBP1, PDGFRA, PDGFRB, PHF6, PIK3C2B, PIK3CA, PIK3CB, PIK3CD, PIK3CG, PIK3R1, PIK3R2, PIM1, PLCG1, PMS2, POLE, PPM1D, PPP2R1A, PPP6C, PRKACA, PTCH1, PTEN, PTPN11, PTPRD, PXDNL, RAC1, RAD50, RAD51, RAF1, RARA, RB1, RET, RGS7, RHEB, RHOA, RICTOR, RIT1, RNF43, ROS1, RPL10, RPL5, RUNX1, RUNX1T1, SDHD, SETBP1, SETD2, SF3B1, SIX1, SIX2, SLCO1B3, SLX4, SMAD2, SMAD4, SMARCA4, SMARCB1, SMC1A, SMO, SNCAIP, SOCS1, SOS1, SOX2, SPOP, SRC, SRSF2, STAG2, STAT3, STAT5B, STAT6, STK11, TAF1, TCF7L2, TERT, TET2, TGFBR1, TGFBR2, TNFAIP3, TOP1, TP53, TPMT, TRRAP, TSC2, TSHR, U2AF1, UGT1A1, USP8, VHL, WAS, WT1, XPO1, XRCC2, ZFHX3, ZNF217, ZNF429')
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)   
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed for the Detection of Copy Number Variations: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('ABCB1, ABL1, ABL2, ABRAXAS1, ACVR1B, ACVR2A, ADAMTS12, ADAMTS2, AKT1, AKT2, AKT3, ALK, AMER1, APC, AR, ARAF, ARHGAP35, ARID1A, ARID1B, ARID2, ARID5B, ASXL1, ASXL2, ATM, ATR, ATRX, AURKA, AURKC, AXIN1, AXIN2, AXL, B2M, BAP1, BARD1, BCL2, BCL2L12, BCL6, BCOR, BLM, BMPR2, BRAF, BRCA1, BRCA2, BRIP1, CARD11, CASP8, CBFB, CBL, CCND1, CCND2, CCND3, CCNE1, CD274, CD276, CDC73, CDH1, CDH10, CDK12, CDK4, CDK6, CDKN1A, CDKN1B, CDKN2A, CDKN2B, CDKN2C, CHD4, CHEK1, CHEK2, CIC, CREBBP, CSMD3, CTCF, CTLA4, CTNND2, CUL3, CUL4A, CUL4B, CYLD, CYP2C9, DAXX, DDR1, DDR2, DDX3X, DICER1, DNMT3A, DOCK3, DPYD, DSC1, DSC3, EGFR, EIF1AX, ELF3, EMSY, ENO1, EP300, EPCAM, EPHA2, ERAP1, ERAP2, ERBB2, ERBB3, ERBB4, ERCC2, ERCC4, ERRFI1, ESR1, ETV6, EZH2, FAM135B, FANCA, FANCC, FANCD2, FANCE, FANCF, FANCG, FANCI,  FANCL, FANCM, FAT1, FBXW7, FGF19, FGF23, FGF3, FGF4, FGF9, FGFR1, FGFR2, FGFR3, FGFR4, FLT3, FLT4, FOXA1, FUBP1, FYN, GATA2, GATA3, GLI3, GNA13, GNAS, GPS2, H3-3A, H3-3B, HDAC2, HDAC9, HLA-A, HLA-B, HNF1A, IDH2, IGF1R, IKBKB, IL7R, INPP4B, JAK1, JAK2, JAK3, KDM5C, KDM6A, KDR, KEAP1, KIT, KLF5, KMT2A, KMT2B, KMT2C, KMT2D, KRAS, LARP4B, LATS1, LATS2, MAGOH, MAP2K1, MAP2K4, MAP2K7, MAP3K1, MAP3K4, MAPK1, MAPK8, MAX, MCL1, MDM2, MDM4, MECOM, MEF2B, MEN1, MET, MGA, MITF, MLH1, MLH3, MPL, MRE11, MSH2, MSH3, MSH6, MTAP, MTOR, MUTYH, MYC, MYCL, MYCN, MYD88, NBN, NCOR1, NF1, NF2, NFE2L2, NOTCH1, NOTCH2, NOTCH3, NOTCH4, NRAS, NTRK1, NTRK3, PALB2, PARP1, PARP2, PARP3, PARP4, PBRM1, PCBP1, PDCD1, PDCD1LG2, PDGFRA, PDGFRB, PDIA3, PGD, PHF6, PIK3C2B, PIK3CA, PIK3CB, PIK3R1, PIK3R2, PIM1, PLCG1, PMS1, PMS2, POLD1, POLE, POT1, PPM1D, PPP2R1A, PPP2R2A, PPP6C, PRDM1, PRDM9, PRKACA, PRKAR1A, PTCH1, PTEN, PTPN11, PTPRT, PXDNL, RAC1, RAD50, RAD51, RAD51B, RAD51C, RAD51D, RAD52, RAD54L, RAF1, RARA, RASA1, RASA2, RB1, RBM10, RECQL4, RET, RHEB, RICTOR, RIT1, RNASEH2A, RNASEH2B, RNF43, ROS1, RPA1, RPS6KB1, RPTOR, RUNX1, SDHA, SDHB, SDHD, SETBP1, SETD2, SF3B1, SLCO1B3, SLX4, SMAD2, SMAD4, SMARCA4, SMARCB1, SMC1A, SMO, SOX9, SPEN, SPOP, SRC, STAG2, STAT3, STAT6, STK11, SUFU, TAP1, TAP2, TBX3, TCF7L2, TERT, TET2, TGFBR2, TNFAIP3, TNFRSF14, TOP1, TP53, TP63, TPMT, TPP2, TSC1, TSC2, U2AF1, USP8, USP9X, VHL, WT1, XPO1, XRCC2, XRCC3, YAP1, YES1, ZFHX3, ZMYM3, ZNF217, ZNF429, ZRSR2')    
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed for the Detection of Fusions: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('AKT1, AKT2, AKT3, ALK, AR, BRAF, BRCA1, CDKN2A, EGFR, ERBB2, ERBB4, ERG, ESR1, ETV1, ETV4, ETV5, FGFR1, FGFR2, FGFR3, MAP3K8, MET, MTAP, MYB, MYBL1, NOTCH1, NOTCH2, NOTCH3, NRG1, NTRK1, NTRK2, NTRK3, NUTM1, PIK3CA, PIK3CB, PPARG, PRKACA, PRKACB, RAF1, RARA, RELA, RET, ROS1, RSPO2, RSPO3, STAT6, TERT, TFE3, TFEB, YAP1')    
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed with Full Exon Coverage: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('ABRAXAS1, ACVR1B, ACVR2A, ADAMTS12, ADAMTS2, AMER1, APC, ARHGAP35, ARID1A, ARID1B, ARID2, ARID5B, ASXL1, ASXL2, ATM, ATR, ATRX, AXIN1, AXIN2, B2M, BAP1, BARD1, BCOR, BLM, BMPR2, BRCA1, BRCA2, BRIP1, CALR, CASP8, CBFB, CD274, CD276, CDC73, CDH1, CDH10, CDK12, CDKN1A, CDKN1B, CDKN2A, CDKN2B, CDKN2C, CHEK1, CHEK2, CIC, CIITA, CREBBP, CSMD3, CTCF, CTLA4, CUL3, CUL4A, CUL4B, CYLD, CYP2C9, CYP2D6, DAXX, DDX3X, DICER1, DNMT3A, DOCK3, DPYD, DSC1, DSC3, ELF3, ENO1, EP300, EPCAM, EPHA2, ERAP1, ERAP2, ERCC2, ERCC4, ERCC5, ERRFI1, ETV6, FANCA, FANCC, FANCD2, FANCE, FANCF, FANCG, FANCI, FANCL, FANCM, FAS, FAT1, FBXW7, FUBP1, GATA3, GNA13, GPS2, HDAC2, HDAC9, HLA-A, HLA-B, HNF1A, ID3, INPP4B, JAK1, JAK2, JAK3, KDM5C, KDM6A, KEAP1, KLHL13, KMT2A, KMT2B, KMT2C, KMT2D, LARP4B, LATS1, LATS2, MAP2K4, MAP2K7, MAP3K1, MAP3K4, MAPK8, MEN1, MGA, MLH1, MLH3, MRE11, MSH2, MSH3, MSH6, MTAP, MTUS2, MUTYH, NBN, NCOR1, NF1, NF2, NOTCH1, NOTCH2, NOTCH3, NOTCH4, PALB2, PARP1, PARP2, PARP3, PARP4, PBRM1, PDCD1, PDCD1LG2, PDIA3, PGD, PHF6, PIK3R1, PMS1, PMS2, POLD1, POLE, POT1, PPM1D, PPP2R2A, PRDM1, PRDM9, PRKAR1A, PSMB10, PSMB8, PSMB9, PTCH1, PTEN, PTPRT, RAD50, RAD51, RAD51B, RAD51C, RAD51D, RAD52, RAD54L, RASA1, RASA2, RB1, RBM10, RECQL4, RNASEH2A, RNASEH2B, RNASEH2C, RNF43, RPA1, RPL22, RPL5, RUNX1, RUNX1T1, SDHA, SDHB, SDHC, SDHD, SETD2, SLX4, SMAD2, SMAD4, SMARCA4, SMARCB1, SOCS1, SOX9, SPEN, STAG2, STAT1, STK11, SUFU, TAP1, TAP2, TBX3, TCF7L2, TET2, TGFBR2, TMEM132D, TNFAIP3, TNFRSF14, TP53, TP63, TPP2, TSC1, TSC2, UGT1A1, USP9X, VHL, WT1, XRCC2, XRCC3, ZBTB20, ZFHX3, ZMYM3, ZRSR2')    
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)

    # Methodology section
    p = document.add_paragraph()
    new_width = Cm(6.85)
    new_height = Cm(0.69)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Methodology.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Nucleic acid (DNA/RNA) was extracted from FFPE tissue samples, using standard Qiagen nucleic acid isolation kits. The DNA/RNA was quantified using Qubit and 20-30 ng of DNA/RNA was amplified using Oncomine Comprehensive assay plus as per the instruction manual. The QC of the library prepared is checked by Agilent Bioanalyser. The 150-200 bp library size samples are sequenced on Ion S5 platform as per manufacturer protocol.')    
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 

    # Data Analysis section
    p = document.add_paragraph()
    new_width = Cm(7.14)
    new_height = Cm(0.69)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Data_analysis.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The sequence data is processed using the analysis pipeline Ion reporter software 5.20.2.0 as suggested by the vendor. This software can detects and annotates low frequency somatic variants (SNPs, InDels, CNVs) from targeted Ion AmpliSeq™ DNA manual or Ion Chef™ automated libraries, computes automatic tumor cellularity, Loss-of-Heterozygosity, TMB and microsatellite instability (MSI), as well as gene fusions from targeted Ion AmpliSeq™ RNA manual or Ion Chef™ automated libraries, from Oncomine™ Comprehensive Assay Plus run on the Ion 550™ Chip. TMB uses the TMB (Non-Germline Mutations) filter chain with TMB algorithm v4.0. MSI status is computed using a baseline with MSI algorithm v4.0.3. Released with: Ion Reporter™ Software 5.20.2.0 Workflow Version: 3.1. Samples are classified as TMB-High or TMB-low using a cutoff value of 10 mut/mb.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 

    #Interpretation of variants section
    p = document.add_paragraph()
    new_width = Cm(8.23)
    new_height = Cm(0.61)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Interpretation_of_variants.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The interpretation of the report is based on the clinical information provided. The annotation for variants was derived using various disease databases like dbSNP, ClinVar. The population frequency information from 1000 genomes, ExAC, GnomAD was used for the elimination of common variants/polymorphism. For prediction of the possible impact of coding non-synonymous SNVs on the structure and function of protein, PolyPhen-2, MT2 and SIFT score was used. Further Oncomine Reporter software was used for annotating variants with a curated list of relevant labels, guidelines, and global clinical trials. Oncomine Comprehensive genomic profiling assay will analyze across >500 genes(SNVs,Indels,CNVs,Fusions), plus key immuno-oncology research biomarkers like tumor mutational burden (TMB), microsatellite instability (MSI), and Homologous Recombination Repair genes (HRR).') 
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The variant is reported as per the Standards and Guidelines for the Interpretation and Reporting of Sequence Variants in Cancer- A Joint Consensus Recommendation of the Association for Molecular Pathology, American Society of Clinical Oncology, and College of American Pathologists.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 
    p = document.add_paragraph()
    supplementary_table(document)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The transcript used for clinical reporting generally represents the canonical transcript (according to the Ensembl release 37 gene model), which is usually the longest coding transcript with strong/multiple supporting evidence. However, clinically relevant variants annotated in alternate complete coding transcripts could also be reported. The in silico predictions are based on Variant Effect Predictor, Ensembl release 91 (SIFT version - 5.2.2; PolyPhen - 2.2.2); 2019 release from dbNSFPv4.0 and MutationTaster2 based on build NCBI/ Ensembl 66.')
    for run in p.runs:
        run.font.size = Pt(10)
        run.font.name ='Times New Roman' 

In [33]:
#Static text in disclaimer section is created 
def disclaimer(document):
    print('In Disclaimer section')
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(6.00)
    new_height = Cm(0.7)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Disclaimer.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    
    bullet_points = ['This report is based on the sample received in the Lilac Insights laboratory; the analysis is based on the assumption that samples received are representative of the patient demographics mentioned on the test requisition form and the sample tube.',
        'Despite all the necessary precautions and stringency adopted whilst performing DNA tests, the currently available data indicates that the technical error rate associated with all types of DNA analysis, is approximately 2%.',
        'As with all diagnostic tests, the laboratory report must be interpreted in conjunction with the presenting clinical profile of the individual tested and evaluation of all reports. Interpretation of variants is performed based on the current knowledge standards and reporting guidelines. In cases of presence of a VUS, we recommend periodic review of these variants to determine any change in classification based on new published research.',
        'The classification and interpretation of all the variants is carried out based on the current state of scientific knowledge and medical understanding. The results should be correlated clinically.',
        'This report cannot be used for medico legal purposes.']

    for point in bullet_points:
        add_bullet_point(document.add_paragraph(), point)

def add_bullet_point(paragraph, text):
    run = paragraph.add_run(u'\u2022 ') 
    font = run.font
    font.name = 'Times New Roman'
    font.size = Pt(10)
    
    # Set font for the entire paragraph
    paragraph_font = paragraph.style.font
    paragraph_font.name = 'Times New Roman'
    paragraph_font.size = Pt(10)
    
    paragraph.add_run(text)
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    paragraph.paragraph_format.left_indent = Cm(2.5)
    paragraph.paragraph_format.right_indent = Cm(2.5)

In [34]:
def process_oncomine_variants():
    '''
    Before starting the document creation the tsv file has to be read and arranged. Along with that the tools has to be run and included in the 
    respective dataframe.
    '''
    
    global dna_sequence_variants_merged_with_filter_df
    #Input 
    print('Reading input oncomine report')
    readOncomineReport()
    print('Completed reading oncomine report')
    
    #Merge with Filter in and Filtered out 
    print('Merging with Filtered in and Filtered out files')
    merge_filtered_variant_data() 
    print('Completed merging with Filtered in and Filtered out files')
    
    #Calling mutation taster 
    print('Mutation Taster extraction running')
    dna_sequence_variants_merged_with_filter_df = mutation_taster(dna_sequence_variants_merged_with_filter_df)

    overall_predictions = []
    for predictions in dna_sequence_variants_merged_with_filter_df["Mutation_taster_results"]:
        # If predictions is a list and contains 'Deleterious', append 'D' (Deleterious) to overall_predictions
        if isinstance(predictions, list) and any('Deleterious' in str(prediction) for prediction in predictions):
            overall_predictions.append('D')  # Deleterious
        
        # If predictions is a list and is empty, append '-' (no predictions) to overall_predictions
        elif isinstance(predictions, list) and len(predictions) == 0:
            overall_predictions.append('-')  
        
        # If predictions is a list and contains only 'Benign', append 'B' (Benign) to overall_predictions
        elif isinstance(predictions, list) and all('Benign' in str(prediction) for prediction in predictions):
            overall_predictions.append('B')  # Benign
    
    # Assign overall predictions to the DataFrame
    dna_sequence_variants_merged_with_filter_df["MT2"] = overall_predictions
    dna_sequence_variants_merged_with_filter_df["Mutation_taster_results"]
    print('Mutation Taster completed')
    
    #Clinvar running on the fly 
    print('ClinVar extraction')
    clinvar_extraction(dna_sequence_variants_merged_with_filter_df, GCHR)
    print('ClinVar extraction completed')
    #print(dna_sequence_variants_merged_with_filter_df)

In [35]:
doc = Document()
def generate_report():
    print('Document generation started')
    # Adds Header and Footer to the document
    add_header_footer(doc) 
    # Adds 2 table for reference. One Autopopulated and one from the Relevant biomarker data in Oncomine report
    add_internal_reference_table(doc) 
    # Patient Information table is added with the Redmine Data
    patient_information(doc) 
    # Adding Image betweeen the sections
    oncoprecise_image(doc) 
     # Displays Main Tumor Type from the Redmine data
    clinical_history(doc)
    # Report summary section has one line description from Redmine data; table from input Oncomine report and static notes below
    report_summary(doc) 
    # Entire Theraputic summary section is static 
    therapeutic_summary(doc) 
    # Immunotherapy marker section information includes PD-L1 status, tumor mutation burden,  microsatellite instability,and mismatch repair status
    immunotherapy_markers(doc) 
    # Summarizes Immunotherapy marker section and extraction of data from In-house IMMUNOTHERAPIES_DB_FILE file based on tumor type and biomarker criteria
    approved_immunotherapy(doc)
    # Entire intrial_immunotherapy section is static
    intrial_immunotherapy(doc)
    # Table 1 mainly from input Oncomine report and Filtered in and Filtered out files. Table 2 is from Filtered in, Filtered out files and data from on-fly ClinVar
    variant_details(doc)
    #Get the description for altered variant paragraph
    process_dna_altered_df()
    process_gene_fusions_df()
    process_copy_number_variants_df()
    # Each variant present in the input Oncomine report will get information from the in-house database, ACMG clinvar, Mycancergenome in each different sections
    gene_variant_description(doc)
    # This gives the LOH score and table from the input Oncomine report and rest of the data is static
    hrr_genes(doc)
    # Entire clinical trial section is static
    clinical_trial(doc)
    # Except Quality control statistics table (comes from input Oncomine report) the entire supplementary information section is static
    supplementary_information(doc)
    # Entire Disclaimer section is static
    disclaimer(doc)
    #clear_output(wait=True)
    
    # Save the document
    name = ONCOMINE_REPORT.split('_')
    name = ' '.join(name[:2]) 
    full_report_name = f'{name}_Full_report.docx'
    doc.save(full_report_name)

    # Display the message
    message = "<p style='font-size: 24px; font-weight: bold;'>PATIENT REPORT IS READY FOR DOWNLOAD</p>"
    display(Markdown(message))
    
    # Get the absolute path to the file
    file_path = os.path.abspath(full_report_name)

    # Display a direct link to the file
    display(FileLink(file_path, result_html_prefix="Link to the file: "))

In [36]:
#run Redmine for the patient
patient_info=getSampleInformationFromRedmine(REDMINE_PATIENT_ID)
#preprocess the dataframe by running all the on-fly database to generate the report 
process_oncomine_variants()
# Call the function to generate the report
generate_report()

Reading input oncomine report
Completed reading oncomine report
Merging with Filtered in and Filtered out files
Completed merging with Filtered in and Filtered out files
Mutation Taster extraction running
Summary table not found in the response.
Mutation Taster completed
ClinVar extraction
Fetching Clinvar results for 12,25398284,CC,AA...
esearch link:https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=clinvar&term=(((12[Chromosome])%20AND%2025398284[Base%20Position%20for%20Assembly%20GRCh37])%20AND%20CC:AA
No data available in Clinvar for query
Fetching Clinvar results for 17,7579574,TGG,T...
esearch link:https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=clinvar&term=(((17[Chromosome])%20AND%207579574[Base%20Position%20for%20Assembly%20GRCh37])%20AND%20TGG:T
No data available in Clinvar for query
ClinVar extraction completed
Document generation started
Adding Header and Footer
Adding Internal refernce tables
In Patient Information Section
Patient Information tabl

<p style='font-size: 24px; font-weight: bold;'>PATIENT REPORT IS READY FOR DOWNLOAD</p>

/home/ubuntu/jupyter_notebooks/Internal/lilac_reporting/JOSE MULAKKAL_Full_report.docx